<a href="https://colab.research.google.com/github/AbrahamHub/TESIS-EHBG-FACS/blob/main/notebooks/03b_diagnostico_cruzado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03b · Diagnóstico cruzado de checkpoints 03 y 04
**Sólo Colab con GPU. Inferencia diagnóstica; no reentrena ni selecciona modelos.**

Se cruzan imitación/refuerzo × decoder AM/POMO y dos controles uniformes.
72 intentos con las cuatro geometrías ya vistas y semillas 11/29/47.
Son 2.4 horas máximas de inferencia más verificaciones/Drive. Los decoders nativos
pueden terminar antes de 120 s: el tope es común, el trabajo realizado puede diferir.
Se informan fallos, origen y tiempos. No hay pruebas de hipótesis ni escenarios nuevos.
El uso diagnóstico de checkpoints históricos no les concede aceptación para la fase final.


In [1]:
import sys, os, json, shutil, subprocess, importlib.metadata
from pathlib import Path
try:
    import google.colab
except ImportError as exc:
    raise RuntimeError('Por instrucción del autor: ejecutar exclusivamente en Google Colab.') from exc
BANK_REFERENCE = {'bank_id': 'pilot00_280553a33c2ab594', 'protocol_version': 'pilot-00-v4', 'manifest_sha256': 'd8fe8d222f14acb9a32acfc62cdb105f3f364b85d47a4e8c96c7ef58c426fc02', 'acceptance_status': 'ready_for_method_development', 'role': 'development_only', 'validation_run_id': '20260917T021417Z_665a3a34', 'evidence': 'docs/NOTEBOOK_00_VALIDATION.md'}
BANK_ID = BANK_REFERENCE['bank_id']
SUPPORTED_BANK_PROTOCOLS = (BANK_REFERENCE['protocol_version'],)
RESUME_RUN_ID = ''
TRAIN_DEVICE = 'auto'
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
WORK = Path('/content/TESIS_DIAGNOSTICS_03B')
RUNTIME = WORK / 'runtime'
(RUNTIME / 'svrplab').mkdir(parents=True, exist_ok=True)
(RUNTIME / 'tests').mkdir(exist_ok=True)
(RUNTIME / 'svrplab/__init__.py').write_text('"""Métodos y evaluador sellado del banco de tesis."""\n')
os.chdir(RUNTIME)
def version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None
needed = ['numpy', 'pandas', 'matplotlib', 'pytest', 'ortools']
if version('torch') is None:
    raise RuntimeError('Se requiere un runtime Colab con PyTorch instalado; no reemplazar CUDA automáticamente.')
needed.append('ortools')
def check_ortools_version():
    import re
    installed = version('ortools')
    if installed is None:
        return
    match = re.match(r'^(\d+)\.(\d+)(?:\.|$)', installed)
    if match is None or not (9, 11) <= tuple(map(int, match.groups())) < (10, 0):
        raise RuntimeError(f'OR-Tools {installed} fuera del rango >=9.11,<10; usar un runtime compatible sin reemplazar bibliotecas automáticamente.')
check_ortools_version()
missing = [name for name in needed if version(name) is None]
if missing:
    constraints = RUNTIME / 'constraints.txt'
    protected = ['numpy','pandas','matplotlib','scipy','protobuf','torch','pillow']
    constraints.write_text(''.join(f'{name}=={version(name)}\n' for name in protected if version(name)))
    requirements = ['gurobipy>=11,<14' if name == 'gurobipy' else ('ortools>=9.11,<10' if name == 'ortools' else name) for name in missing]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraints), *requirements], check=True)
check_ortools_version()
if version('ortools') is None:
    raise RuntimeError('OR-Tools no está disponible para el profesor de factibilidad.')
print('Python:', sys.version)
print('Dependencias:', {name: version(name) for name in needed})
print('GPU recomendada; la asignación se verificará al importar PyTorch.')


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Dependencias: {'numpy': '2.1.3', 'pandas': '2.2.3', 'matplotlib': '3.10.0', 'pytest': '8.4.2', 'ortools': '9.12.4544'}
GPU recomendada; la asignación se verificará al importar PyTorch.


In [2]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/TESIS-EHBG-FACS')
import hashlib
def verify_before_import(folder):
    folder = Path(folder)
    manifest = json.loads((folder / 'CHECKSUMS.json').read_text())
    actual = {str(f.relative_to(folder)) for f in folder.rglob('*')
              if f.is_file() and f.name != 'CHECKSUMS.json' and '__pycache__' not in f.relative_to(folder).parts}
    if actual != set(manifest):
        raise ValueError('Inventario del banco distinto del manifiesto.')
    for relative, expected in manifest.items():
        file = folder / relative
        if not file.resolve().is_relative_to(folder.resolve()):
            raise ValueError('Ruta de manifiesto fuera del banco.')
        if hashlib.sha256(file.read_bytes()).hexdigest() != expected:
            raise ValueError(f'Hash incorrecto: {relative}')
    return manifest
if BANK_ID != BANK_REFERENCE['bank_id']:
    raise ValueError('BANK_ID difiere de la referencia validada; actualizar la referencia común antes de cambiar banco.')
remote_bank = DRIVE_ROOT / 'banks' / BANK_ID
manifest = verify_before_import(remote_bank)
manifest_sha = hashlib.sha256(json.dumps(manifest, sort_keys=True, separators=(',', ':'),
                                        ensure_ascii=False, allow_nan=False).encode('utf-8')).hexdigest()
if manifest_sha != BANK_REFERENCE['manifest_sha256']:
    raise ValueError('El banco no coincide con el manifiesto aceptado en 00; no importar ni ejecutar.')
acceptance = json.loads((remote_bank / 'acceptance.json').read_text())
if (acceptance.get('status') != BANK_REFERENCE['acceptance_status']
        or acceptance.get('protocol_version') != BANK_REFERENCE['protocol_version']):
    raise ValueError('El acta no coincide con la aceptación esperada del banco.')
BANK = WORK / 'banks' / BANK_ID
if not BANK.exists():
    BANK.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(remote_bank, BANK)
if verify_before_import(BANK) != manifest:
    raise ValueError('La copia local no coincide con Drive.')
shutil.copy2(BANK / 'code/svrplab_pilot.py', RUNTIME / 'svrplab/pilot.py')
print('Banco validado en 00, copiado y hashes verificados:', BANK_ID, manifest_sha)


Mounted at /content/drive
Banco validado en 00, copiado y hashes verificados: pilot00_280553a33c2ab594 d8fe8d222f14acb9a32acfc62cdb105f3f364b85d47a4e8c96c7ef58c426fc02


In [3]:
%%writefile svrplab/search.py
"""Contrato de búsqueda determinista; evaluación estocástica en pilot, sin certificados.

El objetivo es el viaje con una cota uniforme de medias por arco. No representa
el costo óptimo del SVRP ni el costo completo de recurso todavía pendiente.
"""
from __future__ import annotations

import math
import numpy as np

from . import pilot as p

OBJECTIVE = 'conservative_upper_travel_minutes_v1'


class SearchProblem:
    def __init__(self, x, spec, time_scale=1000):
        self.x = x
        self.n = x.n
        self.vehicles = len(x.capacities)
        self.capacities = x.capacities
        self.demands = x.demands
        self.open = np.maximum(x.windows[:, 0], x.appear_times)
        self.close = x.windows[:, 1]
        self.horizon = float(x.windows[0, 1])
        self.distances = p.distances(x)
        self.travel = p.nominal_upper_matrix(x, spec, time_scale) / time_scale
        self.time_scale = time_scale

    def route_stats(self, route, vehicle):
        clock = travel = late = 0.0
        node = 0
        starts = []
        for customer in route:
            leg = float(self.travel[node, customer])
            travel += leg
            clock = max(clock + leg, float(self.open[customer]))
            late += max(0.0, clock - float(self.close[customer]))
            starts.append(clock)
            node = customer
        if route:
            leg = float(self.travel[node, 0])
            clock += leg
            travel += leg
        return {'travel_minutes': travel, 'lateness_minutes': late,
                'depot_overtime_minutes': max(0.0, clock - self.horizon),
                'capacity_excess': max(0.0, float(sum(self.demands[j] for j in route)
                                                - self.capacities[vehicle])),
                'service_starts': starts, 'return_minutes': clock}

    def score(self, routes):
        p.validate_routes(self.x, routes)
        stats = [self.route_stats(route, v) for v, route in enumerate(routes)]
        return (sum(r['lateness_minutes'] + r['depot_overtime_minutes'] for r in stats),
                sum(r['travel_minutes'] for r in stats))

    def result(self, routes, status, elapsed, history, **metadata):
        if not math.isfinite(elapsed) or elapsed < 0:
            raise ValueError('Duración de búsqueda inválida.')
        if routes is not None:
            routes = [[int(j) for j in route] for route in routes]
            violation, cost = self.score(routes)
            metadata.update(search_violation_minutes=violation,
                            search_objective_minutes=cost,
                            conservative_feasible=violation <= 1e-9)
        metadata['objective_scope'] = OBJECTIVE
        return {'routes': routes, 'status': status, 'solve_seconds': float(elapsed),
                'history': history, 'metadata': metadata}


Writing svrplab/search.py


In [4]:
%%writefile svrplab/baselines.py
"""Independent conventional controls and paid construction/local search helpers.

EDD means earliest deadline first with feasible cheapest insertion. It is a
fast control, not a lower bound. OR-Tools Routing uses guided local search on
exactly the shared conservative travel grid. It is a heuristic reference, not
an optimal ceiling. No function reads certificates, scenarios or learned data.
All calls that construct/search routes are to be executed exclusively in Colab.

Primary API: https://developers.google.com/optimization/routing/vrptw and
https://developers.google.com/optimization/routing/routing_options .
"""
from __future__ import annotations

from collections.abc import Mapping
import importlib.metadata
import math
from numbers import Integral, Real
import time

import numpy as np

_TOL = 1e-9
_MAX_INT = 2**60
EDD_DEFAULTS = {'seconds': 120.}
ORTOOLS_DEFAULTS = {'seconds': 120., 'capacity_scale': 1000}


def _config(defaults, config):
    if config is not None and not isinstance(config, Mapping):
        raise ValueError('La configuración debe ser un mapping.')
    unknown = set(config or {}) - set(defaults)
    if unknown:
        raise ValueError(f'Parámetros desconocidos: {sorted(unknown)}')
    result = defaults | dict(config or {})
    seconds = result['seconds']
    if isinstance(seconds, bool) or not isinstance(seconds, Real) or not math.isfinite(seconds) or seconds < 0:
        raise ValueError('seconds debe ser finito y no negativo.')
    result['seconds'] = float(seconds)
    if 'capacity_scale' in result:
        value = result['capacity_scale']
        if isinstance(value, bool) or not isinstance(value, Integral) or value <= 0:
            raise ValueError('capacity_scale debe ser un entero positivo.')
        result['capacity_scale'] = int(value)
    return result


def _seed(seed):
    if isinstance(seed, bool) or not isinstance(seed, Integral) or seed < 0:
        raise ValueError('La semilla debe ser un entero no negativo.')
    return int(seed)


def _route_values(problem, route, vehicle):
    stats = problem.route_stats(route, vehicle)
    return (stats['lateness_minutes'] + stats['depot_overtime_minutes'], stats['travel_minutes'])


def _profile(problem, route):
    """Earliest prefix and latest suffix starts for constant-time insertions."""
    nodes = [0] + list(route) + [0]
    earliest = [0.]
    for previous, customer in zip(nodes[:-2], nodes[1:-1]):
        earliest.append(max(earliest[-1] + float(problem.travel[previous, customer]),
                            float(problem.open[customer])))
    latest = [0.] * len(nodes)
    latest[-1] = problem.horizon
    for position in range(len(nodes) - 2, 0, -1):
        customer, following = nodes[position:position + 2]
        latest[position] = min(float(problem.close[customer]),
                               latest[position + 1] - float(problem.travel[customer, following]))
    feasible = all(earliest[i] <= latest[i] + _TOL for i in range(1, len(nodes) - 1))
    return nodes, earliest, latest, feasible


def construct_insertion(problem, rng, deadline, pheromone=None, alpha=0., beta=2., q0=1.):
    """Complete EDD/RCL construction with hard capacity and paid soft repair.

    The deterministic case processes scarce-capacity customers, then deadlines.
    Stochastic construction samples among the three earliest eligible customers
    and feasible insertion positions, guided by travel and (optional) pheromone.
    If no temporal insertion exists, choose minimum total temporal violation;
    do not drop customers, add vehicles, or declare the instance infeasible.
    """
    routes = [[] for _ in range(problem.vehicles)]
    loads = np.zeros(problem.vehicles)
    remaining = set(range(1, problem.n + 1))
    profiles = [_profile(problem, route) for route in routes]
    while remaining:
        if time.perf_counter() >= deadline:
            return None, 'time_limit'
        availability = {j: int(np.count_nonzero(loads + problem.demands[j] <= problem.capacities + _TOL))
                        for j in remaining}
        if min(availability.values()) == 0:
            return None, 'capacity_dead_end'
        scarce = min(availability.values())
        ordered = sorted((j for j in remaining if availability[j] == scarce),
                         key=lambda j: (float(problem.close[j]), float(problem.open[j]), j))
        deterministic = q0 >= 1. or rng.random() < q0
        customer = ordered[0] if deterministic else ordered[int(rng.integers(min(3, len(ordered))))]
        feasible = []
        for vehicle, route in enumerate(routes):
            if time.perf_counter() >= deadline:
                return None, 'time_limit'
            if loads[vehicle] + problem.demands[customer] > problem.capacities[vehicle] + _TOL:
                continue
            nodes, earliest, latest, valid = profiles[vehicle]
            if not valid:
                continue
            for position in range(len(route) + 1):
                previous, following = nodes[position:position + 2]
                start = max(earliest[position] + float(problem.travel[previous, customer]),
                            float(problem.open[customer]))
                if (start > problem.close[customer] + _TOL
                        or start + problem.travel[customer, following] > latest[position + 1] + _TOL):
                    continue
                delta = float(problem.travel[previous, customer] + problem.travel[customer, following]
                              - problem.travel[previous, following])
                trail = (1. if pheromone is None else
                         math.sqrt(max(1e-12, float(pheromone[previous, customer]))
                                   * max(1e-12, float(pheromone[customer, following]))))
                weight = alpha * math.log(trail) - beta * math.log1p(max(0., delta))
                feasible.append((vehicle, position, delta, weight))
        if feasible:
            if deterministic:
                # ACO exploitation uses its pheromone; alpha=0 is cheapest EDD.
                choice = max(feasible, key=lambda row: (row[3], -row[2], -row[0], -row[1]))
            else:
                log_weights = np.array([row[3] for row in feasible])
                weights = np.exp(log_weights - np.max(log_weights))
                choice = feasible[int(rng.choice(len(feasible), p=weights / weights.sum()))]
            vehicle, position = choice[:2]
        else:
            # Preserve a complete diagnostic solution even when construction
            # cannot respect all windows. Full affected-route scores are paid.
            best = None
            for vehicle, route in enumerate(routes):
                if loads[vehicle] + problem.demands[customer] > problem.capacities[vehicle] + _TOL:
                    continue
                old = _route_values(problem, route, vehicle)
                for position in range(len(route) + 1):
                    if time.perf_counter() >= deadline:
                        return None, 'time_limit'
                    candidate = route[:position] + [customer] + route[position:]
                    new = _route_values(problem, candidate, vehicle)
                    row = (new[0] - old[0], new[1] - old[1], vehicle, position)
                    if best is None or row < best:
                        best = row
            if best is None:
                return None, 'capacity_dead_end'
            vehicle, position = best[2:]
        routes[vehicle].insert(position, customer)
        loads[vehicle] += problem.demands[customer]
        profiles[vehicle] = _profile(problem, routes[vehicle])
        remaining.remove(customer)
    return routes, 'complete'


def improve_feasible_routes(problem, routes, rng, deadline, max_moves=100):
    """Paid first-improving relocate repair/descent; retains the input incumbent.

    Return ``(routes, diagnostics)``. Coverage, capacity and fleet remain hard;
    score is lexicographic (temporal violation, conservative travel). An already
    feasible incumbent therefore stays feasible. Infeasible input is permitted
    for repair, but success is not guaranteed. No external routes/solver used.
    """
    if isinstance(max_moves, bool) or not isinstance(max_moves, Integral) or max_moves < 0:
        raise ValueError('max_moves debe ser entero no negativo.')
    current = [list(route) for route in routes]
    incumbent_score = problem.score(current)  # Common coverage/capacity contract before repair.
    stats = [_route_values(problem, route, v) for v, route in enumerate(current)]
    loads = [float(sum(problem.demands[j] for j in route)) for route in current]
    evaluated = accepted = 0
    termination = 'move_limit'
    for _ in range(max_moves):
        if time.perf_counter() >= deadline:
            termination = 'time_limit'
            break
        positions = [(v, i) for v, route in enumerate(current) for i in range(len(route))]
        # Repair the worst routes first; randomize ties with the method's RNG.
        if rng is not None:
            rng.shuffle(positions)
        positions.sort(key=lambda pair: -stats[pair[0]][0])
        chosen = None
        for origin, offset in positions:
            customer = current[origin][offset]
            reduced = current[origin][:offset] + current[origin][offset + 1:]
            reduced_stats = _route_values(problem, reduced, origin)
            for target, route in enumerate(current):
                if target != origin and loads[target] + problem.demands[customer] > problem.capacities[target] + _TOL:
                    continue
                base = reduced if target == origin else route
                for position in range(len(base) + 1):
                    if time.perf_counter() >= deadline:
                        termination = 'time_limit'
                        break
                    if target == origin and position == offset:
                        continue
                    inserted = base[:position] + [customer] + base[position:]
                    new_stats = _route_values(problem, inserted, target)
                    old_pair = stats[origin]
                    new_pair = new_stats
                    if target != origin:
                        old_pair = tuple(stats[origin][k] + stats[target][k] for k in (0, 1))
                        new_pair = tuple(reduced_stats[k] + new_stats[k] for k in (0, 1))
                    evaluated += 1
                    better = (new_pair[0] < old_pair[0] - _TOL or
                              (abs(new_pair[0] - old_pair[0]) <= _TOL
                               and new_pair[1] < old_pair[1] - _TOL
                               and (old_pair[0] > _TOL or new_pair[0] <= _TOL)))
                    if better:
                        candidate = [list(existing) for existing in current]
                        candidate[origin] = reduced
                        candidate[target] = inserted
                        candidate_score = problem.score(candidate)
                        if (candidate_score >= incumbent_score or
                                (incumbent_score[0] <= _TOL and candidate_score[0] > _TOL)):
                            continue
                        incumbent_score = candidate_score
                        chosen = (origin, target, customer, reduced, reduced_stats, inserted, new_stats)
                        break
                if chosen is not None or termination == 'time_limit':
                    break
            if chosen is not None or termination == 'time_limit':
                break
        if chosen is None:
            if termination != 'time_limit':
                termination = 'relocate_local_optimum'
            break
        origin, target, customer, reduced, reduced_stats, inserted, new_stats = chosen
        if origin != target:
            current[origin], stats[origin] = reduced, reduced_stats
            loads[origin] -= problem.demands[customer]
            loads[target] += problem.demands[customer]
        current[target], stats[target] = inserted, new_stats
        accepted += 1
    return current, {'evaluated_neighbors': evaluated, 'accepted_moves': accepted,
                     'termination': termination, 'operator': 'first_improving_relocate',
                     'objective': 'lexicographic_violation_then_conservative_travel'}


def _finish(problem, routes, started, cfg, seed, method, termination, history=(), **extra):
    score = None if routes is None else problem.score(routes)
    status = 'no_solution' if routes is None else ('feasible' if score[0] <= _TOL else 'conservative_infeasible')
    elapsed = time.perf_counter() - started
    result = problem.result(routes, status, elapsed, list(history), method=method, seed=seed,
                            effective_config=cfg, termination=termination,
                            implementation='independent_conventional_baselines_v2',
                            construction='from_scratch_without_certificates',
                            budget_includes_construction=True, budget_includes_repair=True,
                            surrogate_optimal=False, stochastic_optimal=False,
                            reference_is_bound=False, **extra)
    elapsed = time.perf_counter() - started
    result['solve_seconds'] = elapsed
    result['metadata']['budget_overrun_seconds'] = max(0., elapsed - cfg['seconds'])
    return result


def solve_edd(problem, seed, config=None):
    """Fast deterministic EDD insertion; ends after construction, without LS."""
    started = time.perf_counter()
    cfg, seed = _config(EDD_DEFAULTS, config), _seed(seed)
    routes, reason = construct_insertion(problem, np.random.default_rng(seed), started + cfg['seconds'])
    return _finish(problem, routes, started, cfg, seed, 'edd', reason,
                   seed_affects_search=False, construction_rule='EDD_feasible_cheapest_insertion',
                   budget_policy='upper_limit_fast_control_not_required_to_exhaust')


def _ortools():
    try:
        from ortools.constraint_solver import pywrapcp, routing_enums_pb2
    except ImportError as exc:
        raise RuntimeError('La referencia OR-Tools requiere instalar ortools en Colab.') from exc
    return pywrapcp, routing_enums_pb2


def _integer(value, scale):
    raw = float(value) * scale
    if not math.isfinite(raw) or raw < 0 or raw > _MAX_INT:
        raise ValueError('Recurso fuera del rango entero seguro.')
    rounded = round(raw)
    if not math.isclose(raw, rounded, rel_tol=0., abs_tol=1e-6):
        raise ValueError('El recurso no pertenece a la rejilla declarada; no se redondeará para favorecer al método.')
    return int(rounded)


def solve_ortools_routing(problem, seed, config=None):
    """Fixed-fleet Routing+GLS heuristic reference; no dropped customers."""
    started = time.perf_counter()
    cfg, seed = _config(ORTOOLS_DEFAULTS, config), _seed(seed)
    deadline = started + cfg['seconds']
    if time.perf_counter() >= deadline:
        return _finish(problem, None, started, cfg, seed, 'ortools_routing', 'time_limit')
    pywrapcp, routing_enums = _ortools()
    # The reference pays for the identical conventional feasible seed; its
    # absence never substitutes a certificate or changes the fixed fleet.
    routes, construction_reason = construct_insertion(problem, np.random.default_rng(seed), deadline)
    feasible_seed = routes is not None and problem.score(routes)[0] <= _TOL
    seed_score = None if routes is None else list(problem.score(routes))
    # An infeasible EDD diagnostic is not a Routing solution. Retain its score
    # for auditing, but never return its routes under the OR-Tools method name.
    routes = routes if feasible_seed else None
    origin = 'edd_paid_fallback' if feasible_seed else 'none'

    def finish(termination, **extra):
        return _finish(problem, routes, started, cfg, seed, 'ortools_routing', termination,
                       construction_reason=construction_reason,
                       edd_diagnostic_score=seed_score,
                       conventional_seed_used=bool(feasible_seed),
                       solution_origin=origin,
                       returned_paid_edd_fallback=(origin == 'edd_paid_fallback'),
                       seed_affects_search=False, **extra)

    if time.perf_counter() >= deadline:
        return finish('time_limit')
    scale = problem.time_scale
    if isinstance(scale, bool) or not isinstance(scale, Integral) or scale <= 0:
        raise ValueError('time_scale debe ser entero positivo.')
    matrix = [[_integer(value, scale) for value in row] for row in problem.travel]
    horizon = math.floor(float(problem.horizon) * scale)
    demands = [_integer(value, cfg['capacity_scale']) for value in problem.demands]
    capacities = [_integer(value, cfg['capacity_scale']) for value in problem.capacities]
    windows = [(math.ceil(float(problem.open[j]) * scale), min(horizon, math.floor(float(problem.close[j]) * scale)))
               for j in range(problem.n + 1)]
    if problem.vehicles < 1 or any(lo > hi for lo, hi in windows[1:]):
        return finish('incompatible_integer_model')
    manager = pywrapcp.RoutingIndexManager(problem.n + 1, problem.vehicles, 0)
    routing = pywrapcp.RoutingModel(manager)
    transit = routing.RegisterTransitCallback(lambda a, b: matrix[manager.IndexToNode(a)][manager.IndexToNode(b)])
    routing.SetArcCostEvaluatorOfAllVehicles(transit)
    routing.AddDimension(transit, horizon, horizon, True, 'Time')
    times = routing.GetDimensionOrDie('Time')
    for customer in range(1, problem.n + 1):
        times.CumulVar(manager.NodeToIndex(customer)).SetRange(*windows[customer])
    for vehicle in range(problem.vehicles):
        times.CumulVar(routing.Start(vehicle)).SetRange(0, 0)
        times.CumulVar(routing.End(vehicle)).SetRange(0, horizon)
    demand = routing.RegisterUnaryTransitCallback(lambda a: demands[manager.IndexToNode(a)])
    routing.AddDimensionWithVehicleCapacity(demand, 0, capacities, True, 'Capacity')
    params = pywrapcp.DefaultRoutingSearchParameters()
    params.first_solution_strategy = routing_enums.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
    params.local_search_metaheuristic = routing_enums.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    params.log_search = False
    # Import/build/seed/extraction are all part of the same end-to-end budget.
    remaining = deadline - time.perf_counter()
    if remaining <= 0:
        return finish('time_limit')
    params.time_limit.seconds = int(remaining)
    params.time_limit.nanos = int((remaining - int(remaining)) * 1e9)
    routing.CloseModelWithParameters(params)
    assignment = None
    if feasible_seed:
        assignment = routing.ReadAssignmentFromRoutes(routes, True)
    remaining = deadline - time.perf_counter()
    if remaining <= 0:
        return finish('time_limit')
    params.time_limit.seconds = int(remaining)
    params.time_limit.nanos = int((remaining - int(remaining)) * 1e9)
    assignment = (routing.SolveFromAssignmentWithParameters(assignment, params) if assignment is not None
                  else routing.SolveWithParameters(params))
    if assignment is not None:
        candidate = []
        for vehicle in range(problem.vehicles):
            index, route = routing.Start(vehicle), []
            while not routing.IsEnd(index):
                customer = manager.IndexToNode(index)
                if customer:
                    route.append(customer)
                index = assignment.Value(routing.NextVar(index))
            candidate.append(route)
        candidate_score = problem.score(candidate)
        if candidate_score[0] > _TOL:
            raise AssertionError('OR-Tools no satisface el contrato conservador compartido.')
        if routes is None or candidate_score <= problem.score(routes):
            routes = candidate
            origin = 'ortools_routing_assignment'
    termination = ('time_limit' if time.perf_counter() >= deadline else
                   ('routing_no_assignment' if assignment is None else 'routing_search_finished'))
    return finish(termination,
                   solver_status=int(routing.status()), ortools_version=importlib.metadata.version('ortools'),
                   routing_search='parallel_cheapest_insertion_plus_guided_local_search',
                   routing_assignment_found=assignment is not None,
                   capacity_scale=cfg['capacity_scale'], time_scale=int(scale),
                   bound_scope='no_optimality_bound_from_routing_search')


Writing svrplab/baselines.py


In [5]:
%%writefile svrplab/metaheuristics.py
"""Transparent classical comparators for the conservative fixed-fleet problem.

Own ACO and Tabu variants, not official reproductions of a named paper. Both pay
for EDD feasible insertion, repair and local search within their declared wall
clock budget. ACO guides feasible insertions with shared directed pheromone and
improves constructed solutions by relocate descent. Tabu uses relocate, swap,
2-opt and inter-route 2-opt-star. Certificates and scenarios are never read.

max_iterations=None means time-limited search; an integer remains available for
controlled reproducibility tests. Every completed incumbent survives timeout;
unsuccessful constructions are recorded, not labelled proof of infeasibility.
Execute construction/search exclusively in Colab.
"""
from __future__ import annotations

import math
import itertools
import time
from collections.abc import Mapping

import numpy as np

from .search import SearchProblem
from .baselines import construct_insertion, improve_feasible_routes


ACO_DEFAULTS = {
    'seconds': 120., 'max_iterations': None, 'ants': 20,
    'alpha': 1., 'beta': 2., 'evaporation': .1, 'q0': .2,
    'local_search_seconds': 1., 'local_search_moves': 20,
}
TABU_DEFAULTS = {
    'seconds': 120., 'max_iterations': None, 'neighborhood_samples': 80,
    'tenure': 15, 'initial_restarts': 4,
    'local_search_seconds': 1., 'local_search_moves': 20,
}
_TOL = 1e-9


def _configuration(defaults, config):
    if config is not None and not isinstance(config, Mapping):
        raise ValueError('La configuración debe ser un mapping.')
    unknown = set(config or {}) - set(defaults)
    if unknown:
        raise ValueError(f'Parámetros desconocidos: {sorted(unknown)}')
    result = defaults | dict(config or {})
    for name, value in result.items():
        if name == 'max_iterations' and value is None:
            continue
        if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, float, np.integer, np.floating)):
            raise ValueError(f'Parámetro numérico inválido: {name}')
        if not math.isfinite(float(value)):
            raise ValueError(f'Parámetro no finito: {name}')
    for name in ('max_iterations', 'ants', 'neighborhood_samples', 'tenure', 'initial_restarts', 'local_search_moves'):
        if name in result and result[name] is not None:
            if int(result[name]) != result[name] or result[name] < 1:
                raise ValueError(f'{name} debe ser entero positivo.')
            result[name] = int(result[name])
    if result['seconds'] < 0 or result['local_search_seconds'] < 0:
        raise ValueError('Los presupuestos deben ser no negativos.')
    for name in ('alpha', 'beta'):
        if name in result and result[name] < 0:
            raise ValueError(f'{name} debe ser no negativo.')
    if 'evaporation' in result and not 0 < result['evaporation'] <= 1:
        raise ValueError('evaporation debe pertenecer a (0, 1].')
    if 'q0' in result and not 0 <= result['q0'] <= 1:
        raise ValueError('q0 debe pertenecer a [0, 1].')
    return {k: (v if v is None or isinstance(v, int) else float(v)) for k, v in result.items()}


def _seed_value(seed):
    if isinstance(seed, (bool, np.bool_)) or not isinstance(seed, (int, np.integer)) or seed < 0:
        raise ValueError('La semilla debe ser un entero no negativo.')
    return int(seed)


def _capacity_screen(problem):
    """Condiciones necesarias; pasarlas no demuestra que exista partición."""
    if problem.vehicles == 0:
        return problem.n == 0
    return (float(np.sum(problem.demands[1:])) <= float(np.sum(problem.capacities)) + _TOL
            and (problem.n == 0 or float(np.max(problem.demands[1:]))
                 <= float(np.max(problem.capacities)) + _TOL))


def _history_row(started, iteration, score, phase, **extra):
    return {'iteration': int(iteration), 'elapsed_seconds': time.perf_counter() - started,
            'violation_minutes': float(score[0]), 'travel_minutes': float(score[1]),
            'phase': phase, **extra}


def _finish(problem, routes, score, started, config, seed, history, method,
            termination, iterations, **extra):
    elapsed = time.perf_counter() - started
    status = ('no_solution' if routes is None else
              'feasible' if score[0] <= _TOL else 'conservative_infeasible')
    result = problem.result(
        routes, status, elapsed, history, method=method, seed=seed,
        effective_config=config, termination=termination, iterations=int(iterations),
        implementation='project_own_insertion_local_search_v2',
        construction='from_scratch_without_certificates',
        search_objective='lexicographic_temporal_violation_then_conservative_travel_minutes',
        nominal_scope='conservative_upper_mean_matrix_not_stochastic_feasibility',
        budget_includes_construction=True, budget_includes_repair=True,
        initialization='paid_edd_feasible_cheapest_insertion',
        budget_overrun_seconds=max(0., elapsed - config['seconds']),
        time_limit_granularity='construction_step_or_neighborhood_candidate',
        seed_reproducibility='same_environment_and_iteration_cutoff; wall_time_cutoff_may_differ',
        **extra)
    # Incluir también la validación final del contrato común en el tiempo medido.
    elapsed = time.perf_counter() - started
    result['solve_seconds'] = elapsed
    result['metadata']['budget_overrun_seconds'] = max(0., elapsed - config['seconds'])
    return result


def _construct(problem, rng, deadline, pheromone, alpha, beta, q0):
    return construct_insertion(problem, rng, deadline, pheromone, alpha, beta, q0)


def _iterations(limit):
    return itertools.count(1) if limit is None else range(1, limit + 1)


def _improve(problem, routes, rng, deadline, config):
    """Per-construction local search remains inside the shared search deadline."""
    if time.perf_counter() >= deadline or config['local_search_seconds'] <= 0:
        return routes, {'evaluated_neighbors': 0, 'accepted_moves': 0}
    local_deadline = min(deadline, time.perf_counter() + config['local_search_seconds'])
    return improve_feasible_routes(problem, routes, rng, local_deadline,
                                   max_moves=config['local_search_moves'])


def _arcs(routes):
    """Atributos dirigidos con vehículo, sin arco artificial 0→0 para vacíos."""
    return {(v, int(a), int(b)) for v, route in enumerate(routes) if route
            for a, b in zip([0] + list(route), list(route) + [0])}


def _reinforce(pheromone, routes, weight):
    for route in routes:
        if route:
            for a, b in zip([0] + list(route), list(route) + [0]):
                pheromone[a, b] += weight


def solve_aco(problem: SearchProblem, seed, config=None):
    """EDD seed + pheromone-guided feasible insertion + paid relocate descent.

    Rank deposits: global best 1 and iteration best .5 after evaporation.
    The method remains a declared project variant, not a reproduction claim.
    """
    started = time.perf_counter()
    cfg = _configuration(ACO_DEFAULTS, config)
    seed = _seed_value(seed)
    deadline = started + cfg['seconds']
    rng = np.random.default_rng(seed)
    best, best_score, history = None, None, []
    completed_iterations = constructed = capacity_dead_ends = 0
    local_evaluated = local_moves = 0
    termination = 'iteration_limit'
    if not _capacity_screen(problem):
        return _finish(problem, best, best_score, started, cfg, seed, history, 'aco',
                       'capacity_screen_failed', 0, constructed_solutions=0)
    pheromone = np.ones((problem.n + 1, problem.n + 1))
    if time.perf_counter() < deadline:
        routes, reason = _construct(problem, rng, deadline, pheromone, 0., cfg['beta'], 1.)
        if routes is not None:
            routes, diagnostics = _improve(problem, routes, rng, deadline, cfg)
            local_evaluated += diagnostics['evaluated_neighbors']
            local_moves += diagnostics['accepted_moves']
            best, best_score = routes, problem.score(routes)
            constructed += 1
            history.append(_history_row(started, 0, best_score, 'paid_edd_initialization'))
        elif reason == 'capacity_dead_end':
            capacity_dead_ends += 1
    for iteration in _iterations(cfg['max_iterations']):
        if time.perf_counter() >= deadline:
            termination = 'time_limit'
            break
        iteration_best, iteration_score = None, None
        for _ in range(cfg['ants']):
            if time.perf_counter() >= deadline:
                termination = 'time_limit'
                break
            routes, reason = _construct(problem, rng, deadline, pheromone,
                                        cfg['alpha'], cfg['beta'], cfg['q0'])
            if routes is None:
                if reason == 'time_limit':
                    termination = 'time_limit'
                    break
                capacity_dead_ends += 1
                continue
            routes, diagnostics = _improve(problem, routes, rng, deadline, cfg)
            local_evaluated += diagnostics['evaluated_neighbors']
            local_moves += diagnostics['accepted_moves']
            score = problem.score(routes)
            constructed += 1
            if iteration_score is None or score < iteration_score:
                iteration_best, iteration_score = routes, score
            if best_score is None or score < best_score:
                best, best_score = routes, score
                history.append(_history_row(started, iteration, best_score, 'incumbent',
                                            constructed_solutions=constructed))
        if termination == 'time_limit':
            break
        pheromone *= 1. - cfg['evaporation']
        if iteration_best is not None:
            _reinforce(pheromone, iteration_best, .5)
        if best is not None:
            _reinforce(pheromone, best, 1.)
        np.clip(pheromone, 1e-6, 1e6, out=pheromone)
        completed_iterations = iteration
        if best_score is not None:
            history.append(_history_row(started, iteration, best_score, 'iteration',
                                        constructed_solutions=constructed))
    if time.perf_counter() >= deadline:
        termination = 'time_limit'
    return _finish(problem, best, best_score, started, cfg, seed, history, 'aco',
                   termination, completed_iterations, constructed_solutions=constructed,
                   capacity_dead_ends=capacity_dead_ends,
                   local_search_evaluated_neighbors=local_evaluated, local_search_accepted_moves=local_moves,
                   construction_rule='EDD_RCL_feasible_insertion_with_pheromone',
                   pheromone_rule='global_best_1_plus_iteration_best_0.5_after_evaporation')


def _apply_move(routes, kind, first, second):
    """Transformación pura; posiciones de relocate se refieren tras extracción."""
    candidate = [list(route) for route in routes]
    a, i = first
    b, j = second
    if kind == 'relocate':
        customer = candidate[a].pop(i)
        candidate[b].insert(j, customer)
    elif kind == 'swap':
        candidate[a][i], candidate[b][j] = candidate[b][j], candidate[a][i]
    elif kind == '2opt_star':
        if a == b:
            raise ValueError('2-opt-star requiere rutas distintas.')
        candidate[a], candidate[b] = candidate[a][:i] + candidate[b][j:], candidate[b][:j] + candidate[a][i:]
    elif kind == '2opt':
        if a != b:
            raise ValueError('2-opt requiere una sola ruta.')
        lo, hi = sorted((i, j))
        candidate[a][lo:hi + 1] = reversed(candidate[a][lo:hi + 1])
    else:
        raise ValueError(f'Movimiento desconocido: {kind}')
    return candidate


def _random_neighbor(routes, rng):
    positions = [(v, i) for v, route in enumerate(routes) for i in range(len(route))]
    if not positions:
        return None
    kind = ('relocate', 'swap', '2opt', '2opt_star')[int(rng.integers(4))]
    if kind == 'relocate':
        first = positions[int(rng.integers(len(positions)))]
        target = int(rng.integers(len(routes)))
        target_length = len(routes[target]) - int(target == first[0])
        second = (target, int(rng.integers(target_length + 1)))
    elif kind == '2opt_star':
        if len(routes) < 2:
            return None
        a, b = map(int, rng.choice(len(routes), size=2, replace=False))
        first = (a, int(rng.integers(len(routes[a]) + 1)))
        second = (b, int(rng.integers(len(routes[b]) + 1)))
    elif kind == 'swap':
        if len(positions) < 2:
            return None
        left, right = rng.choice(len(positions), size=2, replace=False)
        first, second = positions[int(left)], positions[int(right)]
    else:
        eligible = [v for v, route in enumerate(routes) if len(route) >= 2]
        if not eligible:
            return None
        vehicle = int(rng.choice(eligible))
        left, right = rng.choice(len(routes[vehicle]), size=2, replace=False)
        first, second = (vehicle, int(left)), (vehicle, int(right))
    candidate = _apply_move(routes, kind, first, second)
    return None if candidate == routes else (candidate, kind)


def _tabu_admissible(added_arcs, tabu_until, iteration, score, best_score):
    # Aspiración estricta: únicamente superar al mejor global lexicográfico.
    return score < best_score or not any(iteration < tabu_until.get(arc, 0) for arc in added_arcs)


def _capacity_valid(problem, routes):
    return all(sum(float(problem.demands[j]) for j in route) <= problem.capacities[v] + _TOL
               for v, route in enumerate(routes))


def solve_tabu(problem: SearchProblem, seed, config=None):
    """Búsqueda tabú con multiaranque propio y vecinos de capacidad válida.

    Acepta deterioro y violaciones temporales para explorar. Un arco eliminado
    queda prohibido para reinserción durante las próximas ``tenure`` iteraciones;
    así la reversión de relocate, swap y 2-opt es tabú, salvo aspiración. Si el
    muestreo no encuentra movimiento admisible se avanza la iteración (expira la
    memoria) y se vuelve a muestrear, sin declarar óptimo local ni infactibilidad.
    """
    started = time.perf_counter()
    cfg = _configuration(TABU_DEFAULTS, config)
    seed = _seed_value(seed)
    deadline = started + cfg['seconds']
    rng = np.random.default_rng(seed)
    best, best_score, history = None, None, []
    restarts = capacity_dead_ends = iterations = accepted = evaluated = 0
    local_evaluated = local_moves = 0
    termination = 'iteration_limit'
    if not _capacity_screen(problem):
        return _finish(problem, best, best_score, started, cfg, seed, history, 'tabu',
                       'capacity_screen_failed', 0, initial_attempts=0)
    # Construcción determinista inicial, seguida de intentos estocásticos;
    # todos se cobran al mismo reloj y ninguno usa una solución de otro método.
    neutral_pheromone = np.ones((problem.n + 1, problem.n + 1))
    for restart in range(cfg['initial_restarts']):
        if time.perf_counter() >= deadline:
            termination = 'time_limit'
            break
        restarts += 1
        routes, reason = _construct(problem, rng, deadline, neutral_pheromone,
                                    alpha=0., beta=2., q0=1. if restart == 0 else .5)
        if routes is None:
            if reason == 'time_limit':
                termination = 'time_limit'
                break
            capacity_dead_ends += 1
            continue
        routes, diagnostics = _improve(problem, routes, rng, deadline, cfg)
        local_evaluated += diagnostics['evaluated_neighbors']
        local_moves += diagnostics['accepted_moves']
        score = problem.score(routes)
        if best_score is None or score < best_score:
            best, best_score = routes, score
            history.append(_history_row(started, 0, best_score, 'initialization', restart=restart))
    if best is None:
        if termination != 'time_limit':
            termination = 'construction_failed'
        return _finish(problem, best, best_score, started, cfg, seed, history, 'tabu',
                       termination, 0, initial_attempts=restarts, capacity_dead_ends=capacity_dead_ends)
    current = [list(route) for route in best]
    current_score = best_score
    tabu_until = {}
    for iteration in _iterations(cfg['max_iterations']):
        if time.perf_counter() >= deadline:
            termination = 'time_limit'
            break
        current_arcs = _arcs(current)
        chosen = chosen_score = chosen_removed = chosen_kind = None
        seen = set()
        for _ in range(cfg['neighborhood_samples']):
            if time.perf_counter() >= deadline:
                termination = 'time_limit'
                break
            proposed = _random_neighbor(current, rng)
            if proposed is None:
                continue
            candidate, kind = proposed
            key = tuple(tuple(route) for route in candidate)
            if key in seen:
                continue
            seen.add(key)
            if not _capacity_valid(problem, candidate):
                continue
            score = problem.score(candidate)
            evaluated += 1
            candidate_arcs = _arcs(candidate)
            added = candidate_arcs - current_arcs
            if not _tabu_admissible(added, tabu_until, iteration, score, best_score):
                continue
            if chosen_score is None or score < chosen_score:
                chosen, chosen_score = candidate, score
                chosen_removed, chosen_kind = current_arcs - candidate_arcs, kind
        if chosen is not None:
            current, current_score = chosen, chosen_score
            accepted += 1
            for arc in chosen_removed:
                tabu_until[arc] = iteration + cfg['tenure'] + 1
            if current_score < best_score:
                best, best_score = [list(route) for route in current], current_score
        tabu_until = {arc: expiry for arc, expiry in tabu_until.items() if expiry > iteration}
        iterations = iteration
        history.append(_history_row(started, iteration, best_score, 'iteration',
                                    current_violation_minutes=float(current_score[0]),
                                    current_travel_minutes=float(current_score[1]),
                                    accepted_move=chosen_kind, evaluated_neighbors=evaluated))
        if termination == 'time_limit':
            break
    if time.perf_counter() >= deadline:
        termination = 'time_limit'
    return _finish(problem, best, best_score, started, cfg, seed, history, 'tabu',
                   termination, iterations, initial_attempts=restarts,
                   capacity_dead_ends=capacity_dead_ends, evaluated_neighbors=evaluated,
                   accepted_moves=accepted, local_search_evaluated_neighbors=local_evaluated,
                   local_search_accepted_moves=local_moves, tabu_rule='removed_directed_vehicle_arcs_forbid_readdition',
                   aspiration='strict_global_best_lexicographic_improvement')


Writing svrplab/metaheuristics.py


In [6]:
%%writefile svrplab/experiments.py
"""Ejecución Colab, evaluación única y checkpoints inmutables para métodos 01–06.

No modifica el banco ni entrega certificados a los algoritmos. Cada resultado
se reevalúa desde sus rutas persistidas con el pilot.py guardado en el banco.
"""
from __future__ import annotations

import csv
import json
import math
import shutil
import time
import uuid
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

from . import pilot as p
from .search import OBJECTIVE, SearchProblem

SCHEMA = 'methods-v2'
SUMMARY_METRICS = [
    'scenarios', 'alpha', 'mean_travel_minutes', 'cvar_travel_minutes',
    'cost_variance_population', 'std_travel_minutes', 'scenario_strict_feasibility',
    'mean_customer_violation_fraction', 'mean_lateness_minutes',
    'mean_depot_overtime_minutes', 'mean_waiting_minutes', 'mean_base_minutes',
    'mean_net_execution_delta_minutes', 'mean_late_customers', 'mean_recourse_reorders',
    'used_vehicles', 'evaluation_seconds', 'trace_validation_seconds',
]


def write_csv(path, rows, fields=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = list(fields or dict.fromkeys(k for row in rows for k in row))
    with path.open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def native(value):
    """Normaliza escalares del evaluador sellado sin modificar su implementación."""
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return [native(item) for item in value.tolist()]
    if isinstance(value, dict):
        return {key: native(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [native(item) for item in value]
    return value


def write_json(path, value):
    p.write_json(path, native(value))


def read_json(path):
    return json.loads(Path(path).read_text())


def validate_scenarios(scenarios, config):
    if set(scenarios) != set(config['scenarios']):
        raise ValueError('Splits del banco incompatibles.')
    seen = set()
    for split, count in config['scenarios'].items():
        seeds = scenarios[split]
        if (len(seeds) != count or len(set(seeds)) != count
                or any(isinstance(s, bool) or not isinstance(s, int) or s < 0 for s in seeds)
                or seen.intersection(seeds)):
            raise ValueError('Escenarios repetidos, solapados o con tamaño/tipo incorrecto.')
        seen.update(seeds)


def validate_config(config):
    if (config['schema_version'] != SCHEMA or config['role'] != 'development_only'
            or config['objective'] != OBJECTIVE or config['scenario_split'] != 'evaluation'
            or config['export_all_scenario_events'] is not True):
        raise ValueError('Contrato experimental incompatible.')
    if not math.isfinite(config['search_seconds']) or config['search_seconds'] <= 0:
        raise ValueError('Presupuesto debe ser positivo y finito.')
    seeds = config['seeds']
    if not seeds or len(set(seeds)) != len(seeds) or any(type(s) is not int or s < 0 for s in seeds):
        raise ValueError('Semillas de métodos inválidas.')
    if type(config['time_scale']) is not int or config['time_scale'] < 1:
        raise ValueError('Escala de tiempo inválida.')


def route_rows(x, routes, context):
    rows = []
    for vehicle, route in enumerate(routes):
        load = 0.0
        for position, node in enumerate([0, *route, 0]):
            load += float(x.demands[node])
            rows.append({**context, 'vehicle': vehicle, 'position': position,
                         'node': node, 'empty_vehicle': not bool(route),
                         'x': float(x.locations[node, 0]), 'y': float(x.locations[node, 1]),
                         'demand': float(x.demands[node]), 'cumulative_load': load,
                         'capacity': float(x.capacities[vehicle]),
                         'window_open': float(x.windows[node, 0]),
                         'window_close': float(x.windows[node, 1]),
                         'appear_time': float(x.appear_times[node])})
    return rows


def validate_trace(x, routes, trace, raw):
    for key, value in raw.items():
        actual = trace[key]
        if isinstance(value, float):
            if not math.isclose(actual, value, rel_tol=1e-12, abs_tol=1e-9):
                raise AssertionError(f'Traza distinta del evaluador: {key}')
        elif actual != value:
            raise AssertionError(f'Traza distinta del evaluador: {key}')
    p.validate_routes(x, trace['realized_routes'])
    if any(sorted(a) != sorted(b) for a, b in zip(routes, trace['realized_routes'])):
        raise AssertionError('El recurso cambió la asignación de vehículo.')
    events = trace['events']
    expected = x.n + sum(bool(r) for r in routes)
    if len(events) != expected:
        raise AssertionError('Eventos incompletos.')
    if any(type(e['vehicle']) is not int or not 0 <= e['vehicle'] < len(routes) for e in events):
        raise AssertionError('Vehículo inválido en eventos.')
    for vehicle, realized in enumerate(trace['realized_routes']):
        vehicle_events = [e for e in events if e['vehicle'] == vehicle]
        path = [0, *realized, 0] if realized else [0]
        if [(e['from'], e['to']) for e in vehicle_events] != list(zip(path, path[1:])):
            raise AssertionError('Arcos distintos de la ruta realizada.')
        previous = 0.0
        for event in vehicle_events:
            node = event['to']
            start = max(event['arrival'], x.windows[node, 0], x.appear_times[node]) if node else event['arrival']
            expected_late = max(0.0, start - x.windows[node, 1])
            if (not all(math.isfinite(event[k]) for k in ('departure', 'arrival', 'service_start', 'late'))
                    or not math.isclose(event['departure'], previous, abs_tol=1e-9)
                    or event['arrival'] < event['departure'] - 1e-9
                    or not math.isclose(event['service_start'], start, abs_tol=1e-9)
                    or not math.isclose(event['late'], expected_late, abs_tol=1e-9)):
                raise AssertionError('Horario de eventos incoherente.')
            previous = event['service_start']
    travel = sum(e['arrival'] - e['departure'] for e in events)
    waiting = sum(e['service_start'] - e['arrival'] for e in events)
    late = sum(e['late'] for e in events if e['to'] != 0)
    overtime = sum(e['late'] for e in events if e['to'] == 0)
    for value, key in [(travel, 'travel_minutes'), (waiting, 'waiting_minutes'),
                       (late, 'lateness_minutes'), (overtime, 'depot_overtime_minutes')]:
        if not math.isclose(value, raw[key], rel_tol=1e-10, abs_tol=1e-7):
            raise AssertionError(f'Eventos no reconcilian {key}.')


def evaluate_solution(x, solution, seeds, spec, policy, alpha, directory, context):
    """Nunca selecciona/tunea con evaluation; puntúa una solución ya congelada."""
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    write_json(directory / 'environment.json', p.environment_report())
    routes = solution['routes']
    before = x.sha
    p.validate_routes(x, routes)
    summary, records = p.evaluate(x, routes, seeds, spec, policy, alpha)
    if [r['scenario_seed'] for r in records] != seeds:
        raise AssertionError('El evaluador cambió el conjunto u orden de escenarios.')
    trace_started = time.perf_counter()
    event_count = route_count = 0
    # Streaming: no acumular millones de eventos en RAM para n=300.
    with (directory / 'events.csv').open('w', newline='', encoding='utf-8') as ef, \
            (directory / 'realized_routes.csv').open('w', newline='', encoding='utf-8') as rf:
        ew = rw = None
        for record in records:
            seed = record['scenario_seed']
            trace = p.simulate(x, routes, seed, spec, policy, trace=True)
            validate_trace(x, routes, trace, record)
            trace_context = {**context, 'policy': policy, 'scenario_seed': str(seed)}
            for event_index, event in enumerate(trace['events']):
                row = {**trace_context, 'event_index': event_index, **event}
                if ew is None:
                    ew = csv.DictWriter(ef, fieldnames=list(row)); ew.writeheader()
                ew.writerow(row)
                event_count += 1
            for row in route_rows(x, trace['realized_routes'], trace_context):
                if rw is None:
                    rw = csv.DictWriter(rf, fieldnames=list(row)); rw.writeheader()
                rw.writerow(row)
                route_count += 1
    summary['trace_validation_seconds'] = time.perf_counter() - trace_started
    costs = [r['travel_minutes'] for r in records]
    for source in ['waiting_minutes', 'base_minutes', 'net_execution_delta_minutes',
                   'late_customers', 'recourse_reorders']:
        summary['mean_' + source] = float(np.mean([r[source] for r in records]))
    summary['std_travel_minutes'] = float(np.std(costs, ddof=0))
    # CVaR por integración de masa en la cola: control independiente de la fórmula de pilot.
    descending = sorted(costs, reverse=True)
    mass = len(costs) * (1 - alpha)
    whole = min(len(costs), int(math.floor(mass)))
    tail_sum = sum(descending[:whole])
    if whole < len(costs):
        tail_sum += (mass - whole) * descending[whole]
    if not math.isclose(tail_sum / mass, summary['cvar_travel_minutes'], rel_tol=1e-10):
        raise AssertionError('CVaR no coincide con integración de la cola.')
    if x.sha != before:
        raise AssertionError('La instancia cambió durante evaluación.')
    raw_rows = [{**context, 'policy': policy, **r, 'scenario_seed': str(r['scenario_seed'])}
                for r in records]
    write_csv(directory / 'scenario_metrics.csv', raw_rows)
    write_json(directory / 'summary.json', {**context, **summary})
    qc = {'status': 'passed', 'structural_valid': True, 'scenarios': len(records),
          'event_rows': event_count, 'realized_route_rows': route_count,
          'expected_event_rows': len(seeds) * (x.n + solution['nominal']['used_vehicles']),
          'expected_realized_route_rows': len(seeds) * (x.n + 2 * len(x.capacities)),
          'instance_unchanged': x.sha == before, 'trace_matches_evaluator': True,
          'recourse_preserves_vehicle_assignment': True, 'cvar_independently_checked': True,
          'nominal_feasible': solution['nominal']['strict_feasible'],
          'stochastic_feasibility_is_a_measurement': True}
    if qc['event_rows'] != qc['expected_event_rows'] or qc['realized_route_rows'] != qc['expected_realized_route_rows']:
        raise AssertionError('Número de filas de exportación incorrecto.')
    # Releer CSVs comprueba la persistencia, las semillas de 64 bits y el número de filas.
    for name, expected_rows in [('events.csv', event_count), ('realized_routes.csv', route_count),
                                ('scenario_metrics.csv', len(seeds))]:
        with (directory / name).open(newline='', encoding='utf-8') as stream:
            rows = csv.DictReader(stream)
            count = 0
            observed = set()
            for row in rows:
                observed.add(int(row['scenario_seed'])); count += 1
            if count != expected_rows or observed != set(seeds):
                raise AssertionError(f'Exportación incompleta: {name}')
    write_json(directory / 'validation.json', qc)
    return {**context, **summary}


def save_plot(x, routes, path, title):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 6))
    for vehicle, route in enumerate(routes):
        if not route:
            continue
        coords = x.locations[[0, *route, 0]]
        ax.plot(coords[:, 0], coords[:, 1], '-o', markersize=2, linewidth=.8,
                label=f'V{vehicle}')
    ax.scatter(*x.locations[0], marker='*', s=160, color='black', zorder=5)
    ax.set(title=title, xlabel='x (unidad fuente)', ylabel='y (unidad fuente)', aspect='equal')
    if sum(bool(r) for r in routes) <= 15:
        ax.legend(fontsize=6, ncol=3)
    fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)


def _commit(directory, remote):
    p.seal_bundle(directory)
    _sync(directory, remote)


def _sync(local, remote):
    """Conservar staging interrumpido; promoverlo sólo si su hash es idéntico."""
    staging = remote.with_name(remote.name + '.partial')
    if not remote.exists() and staging.exists():
        try:
            complete = p.verify_bundle(staging) == p.verify_bundle(local)
        except (ValueError, OSError, json.JSONDecodeError):
            complete = False
        if complete:
            staging.rename(remote)
        else:
            staging.rename(staging.with_name(staging.name + '.interrupted_' + uuid.uuid4().hex[:8]))
    p.sync_verified(local, remote)


def _recover(local, remote):
    if (local / 'CHECKSUMS.json').exists():
        p.verify_bundle(local)
        _sync(local, remote)
        return True
    if remote.exists():
        # Conservar la escritura local incompleta, sin sobrescribirla ni mezclarla.
        if local.exists():
            local.rename(local.with_name(local.name + '.interrupted_' + uuid.uuid4().hex[:8]))
        _sync(remote, local)
        return True
    if local.exists():
        local.rename(local.with_name(local.name + '.interrupted_' + uuid.uuid4().hex[:8]))
    return False


def comparison_protocol(bank_hash, evaluator_hash, bank_config, config):
    """Mismo problema/evaluación/presupuesto; algoritmos y entrenamiento pueden diferir."""
    return {'schema_version': SCHEMA, 'bank_manifest_sha256': bank_hash,
            'evaluator_sha256': evaluator_hash,
            'runner_sha256': p.file_hash(__file__),
            'search_contract_sha256': p.file_hash(Path(__file__).with_name('search.py')),
            'objective': config['objective'], 'time_scale': config['time_scale'],
            'role': config['role'], 'solver_seeds': config['seeds'],
            'search_seconds': config['search_seconds'], 'scenario_split': config['scenario_split'],
            'policies': bank_config['policies'], 'alpha': bank_config['alpha_cvar'],
            'export_all_scenario_events': config['export_all_scenario_events']}


def provenance_columns(provenance):
    fields = ['training_id', 'model_checkpoint_sha256', 'fold_id', 'training_seed',
              'train_geometry_groups', 'validation_geometry_groups', 'heldout_geometry_groups',
              'training_seconds', 'validation_seconds', 'teacher_seconds', 'offline_cost_id',
              'offline_manifest_sha256', 'checkpoint_selection', 'training_scope']
    result = {key: provenance.get(key) for key in fields}
    for key in ['training_seconds', 'validation_seconds', 'teacher_seconds']:
        result[key] = provenance.get(key, 0.0)
        if isinstance(result[key], bool) or not isinstance(result[key], (float, int)) or not math.isfinite(result[key]) or result[key] < 0:
            raise ValueError(f'Costo offline inválido: {key}')
    for key, value in result.items():
        if isinstance(value, (dict, list, tuple)):
            result[key] = json.dumps(value, sort_keys=True, separators=(',', ':'))
    return result


def validate_learned_provenance(provenance, x, seed):
    groups = [provenance.get(role) for role in
              ['train_geometry_groups', 'validation_geometry_groups', 'heldout_geometry_groups']]
    if any(not isinstance(group, list) or not group for group in groups):
        raise ValueError('Faltan grupos de entrenamiento/validación/heldout del modelo.')
    train, validation, heldout = map(set, groups)
    if (train & validation or train & heldout or validation & heldout
            or x.provenance['geometry_group_sha256'] not in heldout
            or provenance.get('training_seed') != seed):
        raise ValueError('Procedencia del modelo tiene fuga de geometría o semilla distinta.')
    checkpoint = provenance.get('model_checkpoint_sha256', '')
    if not provenance.get('training_id') or len(checkpoint) != 64 or any(c not in '0123456789abcdef' for c in checkpoint):
        raise ValueError('Falta identidad/hash de checkpoint del modelo aprendido.')


def run_methods(bank, config, solvers, work_root, drive_root, run_id=None, validation_artifacts=(),
                method_availability=None, method_provenance=None):
    """solvers: {method: callable(problem, seed, config, output_dir)}.

    Rerun con run_id explícito recupera checkpoints; sin ID crea otra réplica
    trazable. La comparación usa comparison_id, sin confundir réplicas con N.
    """
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise RuntimeError('Experimentación exclusivamente en Google Colab.') from exc
    validate_config(config)
    bank = Path(bank)
    instances = sorted(p.load_bank(bank), key=lambda x: x.n)
    bank_config = read_json(bank / 'config.json')
    bank_hash = p.verify_bundle(bank)
    spec = p.Traffic(**bank_config['traffic'])
    nominal_spec = p.Traffic(**(asdict(spec) | {'mode': 'nominal_mean'}))
    scenarios = {x.n: read_json(bank / f'scenarios/n{x.n:04d}.json') for x in instances}
    for value in scenarios.values():
        validate_scenarios(value, bank_config)
    if not solvers or not set(solvers).issubset(config['methods']):
        raise ValueError('Métodos solicitados incompatibles con configuración.')
    method_availability = method_availability or {name: 'enabled' for name in solvers}
    if set(method_availability) != set(solvers) or not set(method_availability.values()).issubset({'enabled', 'license_pending'}):
        raise ValueError('Disponibilidad de métodos inválida.')
    package = Path(__file__).parent
    source_hashes = {f.name: p.file_hash(f) for f in sorted(package.glob('*.py'))}
    method_provenance = method_provenance or {}
    protocol = comparison_protocol(bank_hash, p.file_hash(p.__file__), bank_config, config)
    comparison_id = p.digest(protocol)
    signature = p.digest({'protocol': protocol, 'config': config,
                          'methods': sorted(solvers), 'source_hashes': source_hashes,
                          'method_availability': method_availability, 'method_provenance': method_provenance})
    run_id = run_id or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ_') + uuid.uuid4().hex[:8]
    if Path(run_id).name != run_id or run_id in ('.', '..'):
        raise ValueError('run_id debe ser un nombre de carpeta simple.')
    root = Path(work_root) / 'method_runs' / bank.name / run_id
    remote = Path(drive_root) / 'method_runs' / bank.name / run_id
    print(f'RUN_ID = {run_id}\nDrive: {remote}', flush=True)
    if _recover(root / 'meta', remote / 'meta'):
        if read_json(root / 'meta/plan.json')['signature'] != signature:
            raise ValueError('No se puede reanudar con otro banco/código/protocolo/configuración/disponibilidad de licencia. Use RUN_ID nuevo.')
    else:
        meta = root / 'meta'
        meta.mkdir(parents=True)
        write_json(meta / 'plan.json', {'run_id': run_id, 'signature': signature,
                     'comparison_id': comparison_id, 'protocol': protocol, 'config': config,
                     'methods': sorted(solvers), 'source_hashes': source_hashes,
                     'method_availability': method_availability,
                     'method_provenance': method_provenance,
                     'bank_id': bank.name, 'role': 'development_only',
                     'created_utc': datetime.now(timezone.utc).isoformat()})
        write_json(meta / 'environment.json', p.environment_report())
        for source in package.glob('*.py'):
            destination = meta / 'code' / source.name
            destination.parent.mkdir(exist_ok=True)
            shutil.copy2(source, destination)
        for source in map(Path, validation_artifacts):
            destination = meta / 'validation' / source.name
            destination.parent.mkdir(exist_ok=True)
            shutil.copy2(source, destination)
        write_json(meta / 'metrics_schema.json', metric_schema())
        _commit(meta, remote / 'meta')
    if _recover(root / 'report', remote / 'report'):
        for relative, expected in read_json(root / 'report/bundles.json').items():
            if p.verify_bundle(remote / relative) != expected:
                raise ValueError(f'Checkpoint remoto modificado: {relative}')
            if not _recover(root / relative, remote / relative):
                raise ValueError(f'Checkpoint ausente: {relative}')
        print('Reporte completo recuperado y verificado.', flush=True)
        return root
    summaries, failures = [], []
    for x in instances:
        for method, solver in solvers.items():
            for seed in config['seeds']:
                key = f'{method}/n{x.n:04d}/seed_{seed}'
                job, remote_job = root / key, remote / key
                learned = method_provenance.get(method, {}).get(f'{x.sha}:{seed}', {})
                if method in ('am_supervised', 'pomo', 'hbg_facs', 'hbg_no_pheromone') and not learned:
                    raise ValueError('Falta procedencia del modelo entrenado para este intento.')
                if method in ('am_supervised', 'pomo', 'hbg_facs', 'hbg_no_pheromone'):
                    validate_learned_provenance(learned, x, seed)
                context = {'schema_version': SCHEMA, 'comparison_id': comparison_id,
                    'run_id': run_id, 'bank_id': bank.name, 'bank_manifest_sha256': bank_hash,
                    'evaluator_sha256': p.file_hash(p.__file__), 'method': method,
                    'solver_seed': seed, 'instance_id': x.instance_id, 'instance_sha256': x.sha,
                    'geometry_group_sha256': x.provenance['geometry_group_sha256'],
                    'n_customers': x.n, 'objective_scope': OBJECTIVE,
                    'search_budget_seconds': config['search_seconds'],
                    'scenario_set_sha256': p.digest(scenarios[x.n]['evaluation']),
                    **provenance_columns(learned)}
                print(f'{key}: búsqueda / recuperación...', flush=True)
                solution_dir = job / 'solution'
                if not _recover(solution_dir, remote_job / 'solution'):
                    solution_dir.mkdir(parents=True)
                    started = time.perf_counter()
                    try:
                        problem = SearchProblem(x, spec, config['time_scale'])
                        preparation = time.perf_counter() - started
                        settings = {**config['methods'][method],
                                    'seconds': max(0.0, config['search_seconds'] - preparation)}
                        if settings['seconds'] <= 0:
                            solution = problem.result(None, 'time_limit_no_solution', preparation, [])
                        else:
                            solution = solver(problem, seed, settings, solution_dir)
                        solution['search_total_seconds'] = time.perf_counter() - started
                        solution['preparation_seconds'] = preparation
                        solution['nominal'] = None
                        if solution['routes'] is not None:
                            p.validate_routes(x, solution['routes'])
                            conservative = problem.score(solution['routes'])
                            solution['metadata'].update(search_violation_minutes=conservative[0],
                                search_objective_minutes=conservative[1],
                                conservative_feasible=conservative[0] <= 1e-9)
                            solution['nominal'] = p.simulate(x, solution['routes'], 0, nominal_spec,
                                                             'fixed_order', trace=True)
                            if conservative[0] <= 1e-9 and not solution['nominal']['strict_feasible']:
                                raise AssertionError('Solución conservadora viola el simulador nominal.')
                    except Exception as exc:
                        solution = {'routes': None, 'nominal': None, 'status': 'solver_error',
                                    'solve_seconds': time.perf_counter() - started,
                                    'search_total_seconds': time.perf_counter() - started,
                                    'history': [], 'metadata': {'error_type': type(exc).__name__,
                                    'error': str(exc), 'objective_scope': OBJECTIVE}}
                    write_json(solution_dir / 'solution.json', solution)
                    write_json(solution_dir / 'environment.json', p.environment_report())
                    # Read-back antes de evaluar: la solución persistida es la autoridad.
                    solution = read_json(solution_dir / 'solution.json')
                    write_json(solution_dir / 'identity.json', context)
                    write_csv(solution_dir / 'search_history.csv', solution['history'],
                              None if solution['history'] else ['iteration', 'elapsed_seconds'])
                    if solution['routes'] is not None:
                        write_csv(solution_dir / 'planned_routes.csv', route_rows(x, solution['routes'], context))
                        write_csv(solution_dir / 'nominal_events.csv',
                                  [{**context, **e} for e in solution['nominal']['events']])
                        save_plot(x, solution['routes'], solution_dir / 'routes.png',
                                  f'{method} · n={x.n} · seed={seed} · {solution["status"]}')
                    _commit(solution_dir, remote_job / 'solution')
                solution = read_json(solution_dir / 'solution.json')
                if read_json(solution_dir / 'identity.json') != context:
                    raise ValueError('Checkpoint de solución corresponde a otro intento.')
                meta = solution['metadata']
                base = {**context, 'solver_status': solution['status'],
                        'structural_valid': solution['routes'] is not None,
                        'nominal_feasible': solution['nominal']['strict_feasible'] if solution['nominal'] else None,
                        'conservative_feasible': meta.get('conservative_feasible'),
                        'search_violation_minutes': meta.get('search_violation_minutes'),
                        'search_objective_minutes': meta.get('search_objective_minutes'),
                        'solve_seconds': solution['solve_seconds'],
                        'search_total_seconds': solution['search_total_seconds'],
                        'best_bound': meta.get('best_bound'), 'relative_gap': meta.get('relative_gap'),
                        'surrogate_optimal': meta.get('surrogate_optimal'),
                        'model_load_seconds': meta.get('model_load_seconds', 0.0),
                        'tuning_seconds_this_run': 0.0}
                for policy in bank_config['policies']:
                    result_dir = job / policy
                    if solution['routes'] is None:
                        summary = {**base, 'policy': policy, 'evaluation_status': 'no_solution',
                                   **dict.fromkeys(SUMMARY_METRICS)}
                        failures.append({**context, 'policy': policy, 'status': solution['status'],
                                         'error': meta.get('error', '')})
                    else:
                        print(f'{key}: {policy}, {len(scenarios[x.n]["evaluation"])} escenarios...', flush=True)
                        if not _recover(result_dir, remote_job / policy):
                            evaluate_solution(x, solution, scenarios[x.n]['evaluation'], spec, policy,
                                              bank_config['alpha_cvar'], result_dir, context)
                            _commit(result_dir, remote_job / policy)
                        evaluated = read_json(result_dir / 'summary.json')
                        if any(evaluated.get(k) != v for k, v in context.items()) or evaluated.get('policy') != policy:
                            raise ValueError('Checkpoint de evaluación corresponde a otro intento o política.')
                        summary = {**base, **evaluated,
                                   'evaluation_status': 'validated'}
                    summaries.append(summary)
    if p.verify_bundle(bank) != bank_hash:
        raise AssertionError('El banco fue modificado.')
    report = root / 'report'
    report.mkdir(parents=True, exist_ok=True)
    write_csv(report / 'summary.csv', summaries)
    write_csv(report / 'failures.csv', failures, None if failures else ['method', 'n_customers', 'solver_seed', 'status'])
    write_json(report / 'summary.json', summaries)
    inventory = {'meta': p.verify_bundle(root / 'meta')}
    for x in instances:
        for method in solvers:
            for seed in config['seeds']:
                key = f'{method}/n{x.n:04d}/seed_{seed}'
                for part in ['solution', *bank_config['policies']]:
                    relative = f'{key}/{part}'
                    if (root / relative / 'CHECKSUMS.json').exists():
                        inventory[relative] = p.verify_bundle(root / relative)
                        if p.verify_bundle(remote / relative) != inventory[relative]:
                            raise ValueError(f'Copia Drive distinta: {relative}')
    write_json(report / 'bundles.json', inventory)
    save_summary_plot(summaries, report / 'metrics.png', bank_config['policies'])
    write_json(report / 'validation.json', {'status': 'completed', 'bank_unchanged': True,
        'attempts': len(instances) * len(solvers) * len(config['seeds']),
        'summary_rows': len(summaries), 'unsolved_policy_rows': len(failures),
        'not_a_claim_of_all_routes_feasible': True, 'ready_for_confirmatory_comparison': False})
    _commit(report, remote / 'report')
    print(f'COMPLETADO: exportaciones verificadas en {remote}', flush=True)
    return root


def save_summary_plot(summaries, path, policies):
    import matplotlib.pyplot as plt
    rows = [r for r in summaries if r['evaluation_status'] == 'validated']
    if not rows:
        return
    fig, axes = plt.subplots(len(policies), 2, figsize=(12, 4 * len(policies)), squeeze=False)
    for row, policy in enumerate(policies):
        for method in sorted({r['method'] for r in rows}):
            data = [r for r in rows if r['policy'] == policy and r['method'] == method]
            for column, metric in enumerate(['scenario_strict_feasibility', 'mean_travel_minutes']):
                axes[row, column].scatter([r['n_customers'] for r in data],
                                         [r[metric] for r in data], label=method, alpha=.65)
                axes[row, column].set(xlabel='Clientes (una geometría por tamaño)', ylabel=metric,
                                      title=f'{policy} · todas las semillas')
                axes[row, column].legend()
        axes[row, 0].set_ylim(-.03, 1.03)
    fig.suptitle('Piloto de desarrollo · costo de viaje y factibilidad deben interpretarse juntos')
    fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)


def metric_schema():
    return {'schema_version': SCHEMA, 'objective_scope': OBJECTIVE,
            'cost_scope': 'travel_only_not_full_recourse_cost',
            'scenario_strict_feasibility': 'Fracción de escenarios con cobertura, capacidad, ventanas y retorno válidos.',
            'cvar_travel_minutes': 'CVaR empírico con masa fraccionaria, alpha del banco.',
            'cost_variance_population': 'Varianza de viaje entre escenarios, ddof=0.',
            'mean_net_execution_delta_minutes': 'Viaje ejecutado menos viaje base planeado; puede ser negativo; no es Q óptimo.',
            'relative_gap': 'Gap Gurobi de la misma formulación conservadora; null si no disponible.',
            'search_total_seconds': 'Preparación y llamada al método; incluye construcción, extracción y artefactos propios del solver.',
            'evaluation_seconds': 'Tiempo informado por el evaluador común para Monte Carlo.',
            'trace_validation_seconds': 'Segunda pasada determinista, validación y escritura de trazas.',
            'missing_metrics': 'null en JSON / vacío en CSV cuando no hay solución; no convertir en cero.',
            'scenario_seed': 'Entero de 64 bits serializado en decimal: leer CSV como texto para evitar redondeo.',
            'independent_unit': 'geometry_group_sha256; semillas y escenarios no aumentan N.',
            'training_seconds': 'Costo total del modelo de ese fold/semilla, repetido por política; no sumar filas.',
            'teacher_seconds': 'Etiquetado compartido de geometrías train/validation; costos únicos en manifiesto offline.',
            'offline_cost_id': 'Identifica el inventario de costos únicos de entrenamiento/etiquetado.',
            'comparison_id': 'Mismo banco/evaluador/runner/objetivo/presupuesto/escenarios; no exige algoritmos iguales.',
            'solver_seed': 'En am_supervised/pomo/hbg_facs identifica entrenamiento y búsqueda; no equivale al mismo flujo RNG de ACO.',
            'comparison_keys': ['comparison_id', 'instance_sha256', 'solver_seed', 'policy'],
            'summary_metrics': SUMMARY_METRICS}


Writing svrplab/experiments.py


In [7]:
%%writefile svrplab/attention.py
"""Attention Model supervisado propio para 03; ejecución sólo en Google Colab.

Encoder de conjuntos sin posición y decoder pointer con atención y máscaras
conservadoras. Inspirado en la familia AM de Kool et al. (ICLR 2019), pero NO
reproduce su arquitectura exacta, REINFORCE ni sus resultados. Las etiquetas
proceden de búsquedas de desarrollo, nunca de certificados del banco.

Cada depósito seleccionado cierra el vehículo actual y avanza al siguiente.
La flota mantiene su orden original, incluso con capacidades heterogéneas y
rutas vacías. Las construcciones bloqueadas se registran. Durante inferencia,
la completación opcional explora prefijos desde cero, se declara por separado
y no sustituye rutas por etiquetas. No se consultan escenarios estocásticos.

Fuentes técnicas: https://arxiv.org/abs/1803.08475 y
https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html
"""
from __future__ import annotations

from collections.abc import Mapping
from dataclasses import asdict, dataclass
import math
import time

import numpy as np
import torch
from torch import nn

from .search import SearchProblem

IMPLEMENTATION = 'project_supervised_attention_v2'
INFERENCE_DEFAULTS = {'seconds': 120., 'n_samples': 16, 'permute_samples': False,
                      'completion_search_nodes': 2048}
MASK_VERSION = 'necessary_capacity_reachability_v2'
_TOL = 1e-9
_NODE_FEATURES = 9
_STATE_FEATURES = 6
_ACTION_FEATURES = 5


@dataclass(frozen=True)
class AttentionConfig:
    model_dim: int = 128
    num_heads: int = 8
    num_layers: int = 3
    dropout: float = .1
    tanh_clip: float = 10.

    def __post_init__(self):
        for name in ('model_dim', 'num_heads', 'num_layers'):
            value = getattr(self, name)
            if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)) or value < 1:
                raise ValueError(f'{name} debe ser entero positivo.')
        if self.model_dim % self.num_heads:
            raise ValueError('model_dim debe ser divisible entre num_heads.')
        for name in ('dropout', 'tanh_clip'):
            value = getattr(self, name)
            if (isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, float, np.number))
                    or not math.isfinite(float(value))):
                raise ValueError(f'{name} debe ser un número finito.')
        if not 0 <= self.dropout < 1 or self.tanh_clip <= 0:
            raise ValueError('dropout debe pertenecer a [0,1), tanh_clip debe ser positivo.')

    def to_dict(self):
        return asdict(self)

    @classmethod
    def from_value(cls, config):
        if isinstance(config, cls):
            return config
        if config is None:
            return cls()
        if not isinstance(config, Mapping):
            raise ValueError('La configuración del modelo debe ser un mapping.')
        aliases = {'dim': 'model_dim', 'heads': 'num_heads', 'layers': 'num_layers'}
        values = dict(config)
        for short, full in aliases.items():
            if short in values:
                if full in values:
                    raise ValueError(f'No mezcle {short} y {full}.')
                values[full] = values.pop(short)
        unknown = set(values) - set(cls.__dataclass_fields__)
        if unknown:
            raise ValueError(f'Parámetros de modelo desconocidos: {sorted(unknown)}')
        return cls(**values)


def _node_order(problem, node_order):
    """Permuta sólo clientes; el token de depósito conserva la posición cero."""
    if node_order is None:
        return np.arange(problem.n + 1, dtype=np.int64)
    order = list(node_order)
    if (any(isinstance(j, (bool, np.bool_)) or not isinstance(j, (int, np.integer)) for j in order)
            or sorted(order) != list(range(1, problem.n + 1))):
        raise ValueError('node_order debe ser una permutación de los clientes 1..n, sin depósito.')
    return np.array([0, *order], dtype=np.int64)


def _scales(problem):
    xy = problem.x.locations - problem.x.locations[0]
    coord = max(1., float(np.max(np.abs(xy))))
    horizon = max(1., abs(float(problem.horizon)), float(np.max(np.abs(problem.close))),
                  float(np.max(np.abs(problem.open))))
    capacity = max(1., float(np.max(problem.capacities)))
    return coord, horizon, capacity


def _node_features(problem, order, scales):
    coord, horizon, capacity = scales
    locations = (problem.x.locations - problem.x.locations[0]) / coord
    features = np.column_stack((locations, problem.demands / capacity,
                                problem.open / horizon, problem.close / horizon,
                                problem.x.appear_times / horizon,
                                np.arange(problem.n + 1) == 0,
                                problem.travel[0] / horizon, problem.travel[:, 0] / horizon))
    if not np.isfinite(features).all():
        raise ValueError('Features del modelo no finitos.')
    return np.asarray(features[order], dtype=np.float32)


class AttentionPolicy(nn.Module):
    """Encoder permutation-equivariant y decoder pointer condicionado por flota."""
    def __init__(self, config=None):
        super().__init__()
        self.config = AttentionConfig.from_value(config)
        c = self.config
        self.node_embedding = nn.Linear(_NODE_FEATURES, c.model_dim)
        # Capas construidas separadamente: no comparten pesos ni inicialización.
        self.encoder = nn.ModuleList([
            nn.TransformerEncoderLayer(c.model_dim, c.num_heads,
                                       dim_feedforward=4 * c.model_dim,
                                       dropout=c.dropout, activation='gelu',
                                       batch_first=True, norm_first=True)
            for _ in range(c.num_layers)
        ])
        self.encoder_norm = nn.LayerNorm(c.model_dim)
        self.context = nn.Linear(2 * c.model_dim + _STATE_FEATURES, c.model_dim)
        self.action_embedding = nn.Linear(_ACTION_FEATURES, c.model_dim, bias=False)
        self.glimpse = nn.MultiheadAttention(c.model_dim, c.num_heads,
                                           dropout=c.dropout, batch_first=True)
        self.pointer_query = nn.Linear(c.model_dim, c.model_dim, bias=False)
        self.pointer_key = nn.Linear(c.model_dim, c.model_dim, bias=False)

    def encode(self, features):
        encoded = self.node_embedding(features).unsqueeze(0)
        for layer in self.encoder:
            encoded = layer(encoded)
        return self.encoder_norm(encoded).squeeze(0)

    def next_logits(self, encoded, current_local, state_features, action_features, blocked):
        if bool(blocked.all()):
            raise ValueError('No se calcula softmax de un estado sin acciones factibles.')
        query = self.context(torch.cat((encoded.mean(dim=0), encoded[current_local],
                                        state_features))).view(1, 1, -1)
        candidates = encoded + self.action_embedding(action_features)
        glimpse, _ = self.glimpse(query, candidates.unsqueeze(0), candidates.unsqueeze(0),
                                  key_padding_mask=blocked.unsqueeze(0), need_weights=False)
        logits = (self.pointer_key(candidates) * self.pointer_query(glimpse[0, 0])).sum(dim=-1)
        logits = self.config.tanh_clip * torch.tanh(logits / math.sqrt(self.config.model_dim))
        return logits.masked_fill(blocked, -torch.inf)


def _travel_lower_bound(problem):
    """Admissible all-pairs travel bound, including for nonmetric matrices.

    Intermediate vertices may already be visited: allowing them only relaxes
    the completion problem. Rounding each sum down avoids pruning a feasible
    completion due to numerical overestimation. This cache changes no data.
    """
    cached = getattr(problem, '_attention_travel_bound', None)
    if cached is not None:
        return cached
    lower = np.array(problem.travel, dtype=np.float64, copy=True)
    if not np.isfinite(lower).all() or (lower < 0).any():
        raise ValueError('La anticipación requiere tiempos finitos no negativos.')
    np.fill_diagonal(lower, 0.)
    for intermediate in range(problem.n + 1):
        via = np.nextafter(lower[:, intermediate, None] + lower[None, intermediate, :], -np.inf)
        np.minimum(lower, np.maximum(0., via), out=lower)
    lower.setflags(write=False)
    problem._attention_travel_bound = lower
    return lower


class _State:
    """Estado determinista CPU; sólo las features y la atención pasan al dispositivo."""
    def __init__(self, problem):
        self.problem = problem
        self.vehicle = 0
        self.node = 0
        self.clock = 0.
        self.load = 0.
        self.visited = np.zeros(problem.n + 1, dtype=bool)
        self.visited[0] = True
        self.routes = [[] for _ in range(problem.vehicles)]
        self.lower = _travel_lower_bound(problem)

    def clone(self):
        copied = object.__new__(type(self))
        copied.problem, copied.lower = self.problem, self.lower
        copied.vehicle, copied.node = self.vehicle, self.node
        copied.clock, copied.load = self.clock, self.load
        copied.visited = self.visited.copy()
        copied.routes = [list(route) for route in self.routes]
        return copied

    def _completion_possible_after(self, candidates, remaining, start):
        """Necessary conditions only, never a certificate of completion.

        Each pending customer must fit/reach either the current vehicle after
        the proposed action or a fresh future vehicle. Customers that cannot
        use any future vehicle must fit together in the current residual
        capacity. All conditions relax ordering and hence cannot exclude an
        actually feasible completion. Zero-demand customers remain mandatory.
        """
        p = self.problem
        pending = np.flatnonzero(remaining)
        if not len(candidates) or not len(pending):
            return np.ones(len(candidates), dtype=bool)
        future = p.capacities[self.vehicle + 1:]
        future_start = np.maximum(self.lower[0, pending], p.open[pending])
        future_ok = ((p.demands[pending] <= (float(future.max()) if len(future) else -1.) + _TOL)
                     & (future_start <= p.close[pending] + _TOL)
                     & (future_start + self.lower[pending, 0] <= p.horizon + _TOL))
        residual = p.capacities[self.vehicle] - self.load - p.demands[candidates]
        earliest = np.maximum(start[candidates, None] + self.lower[np.ix_(candidates, pending)],
                              p.open[pending][None, :])
        current_ok = ((p.demands[pending][None, :] <= residual[:, None] + _TOL)
                      & (earliest <= p.close[pending][None, :] + _TOL)
                      & (earliest + self.lower[pending, 0][None, :] <= p.horizon + _TOL))
        selected = candidates[:, None] == pending[None, :]
        individual = (selected | current_ok | future_ok[None, :]).all(axis=1)
        total_left = float(p.demands[pending].sum()) - p.demands[candidates]
        total_capacity = residual + float(future.sum())
        mandatory = (p.demands[pending] * ~future_ok)[None, :] * ~selected
        return (individual & (total_left <= total_capacity + (len(future) + 1) * _TOL)
                & (mandatory.sum(axis=1) <= residual + _TOL))

    @property
    def done(self):
        return self.vehicle == self.problem.vehicles

    def inputs(self, order, scales):
        p = self.problem
        _, horizon, capacity = scales
        remaining = ~self.visited
        arrival = self.clock + p.travel[self.node]
        start = np.maximum(arrival, p.open)
        returned = start + p.travel[:, 0]
        allowed = (remaining & (self.load + p.demands <= p.capacities[self.vehicle] + _TOL)
                   & (start <= p.close + _TOL)
                   & (start + self.lower[:, 0] <= p.horizon + _TOL))
        candidates = np.flatnonzero(allowed)
        allowed[candidates] &= self._completion_possible_after(candidates, remaining, start)
        # Cerrar el último vehículo no puede dejar clientes sin atender, ni de demanda cero.
        can_close = self.clock + p.travel[self.node, 0] <= p.horizon + _TOL
        if self.vehicle == p.vehicles - 1:
            can_close = can_close and not bool(remaining.any())
        elif remaining.any():
            future = p.capacities[self.vehicle + 1:]
            earliest = np.maximum(self.lower[0, remaining], p.open[remaining])
            can_close = (can_close and float(p.demands[remaining].sum()) <= float(future.sum()) + len(future) * _TOL
                         and float(p.demands[remaining].max()) <= float(future.max()) + _TOL
                         and bool((earliest <= p.close[remaining] + _TOL).all())
                         and bool((earliest + self.lower[remaining, 0] <= p.horizon + _TOL).all()))
        allowed[0] = can_close
        residual = float(p.capacities[self.vehicle]) - self.load
        state = np.array([self.clock / horizon, residual / capacity,
                          float(p.capacities[self.vehicle]) / capacity,
                          self.vehicle / max(1, p.vehicles - 1),
                          float(remaining.sum()) / max(1, p.n),
                          float(p.capacities[self.vehicle + 1:].sum()) /
                          max(1., float(p.capacities.sum()))], dtype=np.float32)
        action = np.column_stack((p.travel[self.node] / horizon, start / horizon,
                                  (p.close - start) / horizon, p.demands / capacity,
                                  returned / horizon))
        # El depósito significa regreso y cambio de vehículo, no servicio como cliente.
        action[0] = [p.travel[self.node, 0] / horizon,
                     (self.clock + p.travel[self.node, 0]) / horizon,
                     (p.horizon - self.clock - p.travel[self.node, 0]) / horizon,
                     0., (self.clock + p.travel[self.node, 0]) / horizon]
        return state, np.asarray(action[order], dtype=np.float32), ~allowed[order]

    def advance(self, customer):
        if customer == 0:
            self.vehicle += 1
            self.node, self.clock, self.load = 0, 0., 0.
        else:
            self.clock = max(self.clock + float(self.problem.travel[self.node, customer]),
                             float(self.problem.open[customer]))
            self.load += float(self.problem.demands[customer])
            self.node = customer
            self.visited[customer] = True
            self.routes[self.vehicle].append(customer)


def _tensors(state, order, scales, device):
    context, action, blocked = state.inputs(order, scales)
    return (torch.as_tensor(context, device=device), torch.as_tensor(action, device=device),
            torch.as_tensor(blocked, device=device))


def _prepare(model, problem, device, node_order):
    order = _node_order(problem, node_order)
    scales = _scales(problem)
    inverse = np.empty(problem.n + 1, dtype=np.int64)
    inverse[order] = np.arange(problem.n + 1)
    features = torch.as_tensor(_node_features(problem, order, scales), device=device)
    return model.encode(features), order, inverse, scales


def teacher_loss(model, problem, routes, device, node_order=None):
    """NLL media por decisión; el caller decide train/eval y ejecuta backward.

    Rechaza etiquetas sin cobertura/capacidad o no factibles en el modelo
    conservador. Devuelve (loss, {'decisions', 'nll_sum'}), con nll_sum numérico
    separado del grafo. Incluye un cierre por vehículo y clientes de demanda 0.
    No comprueba procedencia: el registro de profesores debe comprobarla antes.
    """
    violation, _ = problem.score(routes)
    if violation > _TOL:
        raise ValueError('La etiqueta no es factible bajo la matriz conservadora común.')
    encoded, order, inverse, scales = _prepare(model, problem, device, node_order)
    state = _State(problem)
    terms = []
    for route in routes:
        for customer in [*route, 0]:
            context, action, blocked = _tensors(state, order, scales, device)
            label = int(inverse[customer])
            if bool(blocked[label]):
                raise ValueError('Una acción del profesor está prohibida por las máscaras comunes.')
            logits = model.next_logits(encoded, int(inverse[state.node]), context, action, blocked)
            term = -torch.log_softmax(logits, dim=-1)[label]
            if not bool(torch.isfinite(term)):
                raise FloatingPointError('NLL no finita; no continuar el entrenamiento.')
            terms.append(term)
            state.advance(int(customer))
    if not state.done or not bool(state.visited.all()) or not terms:
        raise ValueError('Secuencia del profesor incompleta.')
    total = torch.stack(terms).sum()
    return total / len(terms), {'decisions': len(terms), 'nll_sum': float(total.detach().cpu())}


def _sync(device):
    device = torch.device(device)
    if device.type == 'cuda':
        torch.cuda.synchronize(device)


def _expired(deadline, device):
    if deadline is None:
        return False
    _sync(device)
    return time.perf_counter() >= deadline


def decode(model, problem, device, mode='greedy', generator=None, deadline=None, node_order=None):
    """Construye sin reparar; None significa bloqueo o tiempo, no infactibilidad probada.

    En sample, generator debe corresponder al dispositivo de PyTorch utilizado.
    deadline es un instante de perf_counter, compartido por todos los intentos.
    La sincronización CUDA incluye el trabajo asíncrono en dicho presupuesto.
    """
    if mode not in ('greedy', 'sample'):
        raise ValueError('mode debe ser greedy o sample.')
    was_training = model.training
    model.eval()
    try:
        with torch.inference_mode():
            if _expired(deadline, device):
                return None
            encoded, order, inverse, scales = _prepare(model, problem, device, node_order)
            state = _State(problem)
            while not state.done:
                if _expired(deadline, device):
                    return None
                context, action, blocked = _tensors(state, order, scales, device)
                if bool(blocked.all()):
                    return None
                logits = model.next_logits(encoded, int(inverse[state.node]), context, action, blocked)
                if not bool(torch.isfinite(logits[~blocked]).all()):
                    raise FloatingPointError('Logits no finitos en la inferencia.')
                if mode == 'greedy':
                    local = int(torch.argmax(logits).item())
                else:
                    probabilities = torch.softmax(logits, dim=-1)
                    local = int(torch.multinomial(probabilities, 1, generator=generator).item())
                state.advance(int(order[local]))
            # La validación independiente también se incluye en el tiempo de solve_attention.
            violation, _ = problem.score(state.routes)
            return state.routes if violation <= _TOL else None
    finally:
        model.train(was_training)


class UniformMaskedPolicy(nn.Module):
    """Untrained diagnostic: exactly uniform logits over the common mask."""
    def __init__(self, config=None):
        super().__init__()
        self.config = AttentionConfig.from_value(config)

    def encode(self, features):
        return features.new_zeros((len(features), 1))

    def next_logits(self, encoded, current_local, state_features, action_features, blocked):
        return encoded.new_zeros(len(blocked)).masked_fill(blocked, -torch.inf)


def policy_completion_search(model, problem, device, deadline, max_nodes=2048):
    """Policy-ordered depth-first completion, separately declared from a rollout.

    Restarts from the empty state, backtracks only through the shared action
    mask, and shares the caller's deadline. No external routes or solver are
    consulted. These search trajectories have no rollout probability and must
    not be added to on-policy training or replay as sampled trajectories.
    """
    if type(max_nodes) is not int or max_nodes < 0:
        raise ValueError('max_nodes debe ser entero no negativo.')
    stats = {'expanded_nodes': 0, 'backtracks': 0, 'termination': 'node_limit',
             'construction': 'policy_ordered_depth_first_completion',
             'is_policy_sample': False, 'mask_version': MASK_VERSION}
    was_training = model.training
    model.eval()
    try:
        with torch.inference_mode():
            if max_nodes == 0 or _expired(deadline, device):
                stats['termination'] = 'time_limit' if max_nodes else 'disabled'
                return None, stats
            encoded, order, inverse, scales = _prepare(model, problem, device, None)
            stack = [(_State(problem), None)]
            while stack:
                if _expired(deadline, device):
                    stats['termination'] = 'time_limit'
                    return None, stats
                state, choices = stack.pop()
                if state.done:
                    if bool(state.visited.all()) and problem.score(state.routes)[0] <= _TOL:
                        stats['termination'] = 'feasible_found'
                        return state.routes, stats
                    stats['backtracks'] += 1
                    continue
                if choices is None:
                    if stats['expanded_nodes'] >= max_nodes:
                        return None, stats
                    stats['expanded_nodes'] += 1
                    context, action, blocked = _tensors(state, order, scales, device)
                    if bool(blocked.all()):
                        stats['backtracks'] += 1
                        continue
                    logits = model.next_logits(encoded, int(inverse[state.node]), context, action, blocked)
                    if not bool(torch.isfinite(logits[~blocked]).all()):
                        raise FloatingPointError('Logits no finitos en búsqueda de completación.')
                    choices = [int(order[local]) for local in torch.argsort(
                        logits, descending=True, stable=True).cpu().tolist() if not bool(blocked[local])]
                if len(choices) > 1:
                    stack.append((state, choices[1:]))
                child = state.clone()
                child.advance(choices[0])
                stack.append((child, None))
            if not stack:
                stats['termination'] = 'tree_exhausted_under_mask'
            return None, stats
    finally:
        model.train(was_training)


def _inference_config(config):
    if config is not None and not isinstance(config, Mapping):
        raise ValueError('La configuración de inferencia debe ser un mapping.')
    unknown = set(config or {}) - set(INFERENCE_DEFAULTS)
    if unknown:
        raise ValueError(f'Parámetros de inferencia desconocidos: {sorted(unknown)}')
    result = INFERENCE_DEFAULTS | dict(config or {})
    seconds = result['seconds']
    if (isinstance(seconds, (bool, np.bool_)) or not isinstance(seconds, (int, float, np.number))
            or not math.isfinite(float(seconds)) or seconds < 0):
        raise ValueError('seconds debe ser finito y no negativo.')
    samples = result['n_samples']
    if isinstance(samples, (bool, np.bool_)) or not isinstance(samples, (int, np.integer)) or samples < 0:
        raise ValueError('n_samples debe ser entero no negativo.')
    if not isinstance(result['permute_samples'], bool):
        raise ValueError('permute_samples debe ser booleano.')
    nodes = result['completion_search_nodes']
    if type(nodes) is not int or nodes < 0:
        raise ValueError('completion_search_nodes debe ser entero no negativo.')
    return result | {'seconds': float(seconds), 'n_samples': int(samples)}


def solve_attention(problem: SearchProblem, seed, config, model, device):
    """Un greedy y hasta n_samples, todos dentro del mismo límite de inferencia."""
    if isinstance(seed, (bool, np.bool_)) or not isinstance(seed, (int, np.integer)) or seed < 0:
        raise ValueError('La semilla debe ser un entero no negativo.')
    effective = _inference_config(config)
    device = torch.device(device)
    _sync(device)
    started = time.perf_counter()
    deadline = started + effective['seconds']
    generator = torch.Generator(device=device).manual_seed(int(seed) % (2**63 - 1))
    rng = np.random.default_rng(int(seed))
    best, best_score, history = None, (math.inf, math.inf), []
    attempts = completed = failed = 0
    termination = 'sample_limit'
    completion_search = {'termination': 'not_needed', 'expanded_nodes': 0, 'is_policy_sample': False}
    for attempt in range(effective['n_samples'] + 1):
        if _expired(deadline, device):
            termination = 'time_limit'
            break
        mode = 'greedy' if attempt == 0 else 'sample'
        order = (rng.permutation(np.arange(1, problem.n + 1))
                 if mode == 'sample' and effective['permute_samples'] else None)
        attempts += 1
        routes = decode(model, problem, device, mode=mode, generator=generator,
                        deadline=deadline, node_order=order)
        if routes is None:
            failed += 1
        else:
            score = problem.score(routes)
            completed += 1
            if score < best_score:
                best, best_score = routes, score
                _sync(device)
                history.append({'iteration': attempt, 'elapsed_seconds': time.perf_counter() - started,
                                'violation_minutes': float(score[0]), 'travel_minutes': float(score[1]),
                                'phase': mode})
        if _expired(deadline, device):
            termination = 'time_limit'
            break
    if best is None and effective['completion_search_nodes'] and not _expired(deadline, device):
        best, completion_search = policy_completion_search(
            model, problem, device, deadline, effective['completion_search_nodes'])
        if best is not None:
            best_score = problem.score(best)
            history.append({'iteration': attempts, 'elapsed_seconds': time.perf_counter() - started,
                            'violation_minutes': float(best_score[0]), 'travel_minutes': float(best_score[1]),
                            'phase': 'policy_completion_search'})
    if best is None and completion_search['termination'] == 'not_needed':
        completion_search['termination'] = ('no_remaining_budget' if effective['completion_search_nodes'] else 'disabled')
    if _expired(deadline, device):
        termination = 'time_limit'
    _sync(device)
    elapsed = time.perf_counter() - started
    result = problem.result(
        best, 'feasible' if best is not None else 'no_solution', elapsed, history,
        method='am_supervised', seed=int(seed), implementation=IMPLEMENTATION,
        model_config=model.config.to_dict(), effective_config=effective,
        termination=termination, attempts=attempts, completed_decodes=completed, failed_decodes=failed,
        raw_decode_complete_fraction=completed / attempts if attempts else None,
        completion_search=completion_search, mask_version=MASK_VERSION,
        solution_origin='policy_completion_search' if completion_search['termination'] == 'feasible_found' else
                        'raw_policy_decode' if best is not None else 'none',
        inference_device=str(device), budget_includes_construction=True,
        budget_includes_encoding=True, budget_overrun_seconds=max(0., elapsed - effective['seconds']),
        time_limit_granularity='synchronized_decoder_step_or_completed_validation',
        search_objective='conservative_upper_travel_minutes',
        nominal_scope='conservative_upper_mean_matrix_not_stochastic_feasibility',
        construction='from_scratch_without_certificates_or_teacher_fallback',
        seed_reproducibility='same_environment_and_sample_cutoff; wall_time_cutoff_may_differ',
        no_solution_scope='decoder_dead_end_or_budget_not_infeasibility_proof',
        training_included_in_solve_seconds=False)
    _sync(device)
    elapsed = time.perf_counter() - started
    result['solve_seconds'] = elapsed
    result['metadata']['budget_overrun_seconds'] = max(0., elapsed - effective['seconds'])
    return result


Writing svrplab/attention.py


In [8]:
%%writefile svrplab/teacher_feasibility.py
"""Profesor de factibilidad independiente para 03; ejecutar sólo en Colab.

CP-SAT construye rutas desde cero sobre el mismo SearchProblem conservador.
No consulta certificados, solvers del certificador, escenarios ni rutas/hints
externos. No optimiza costo: OPTIMAL de CP-SAT en un modelo sin objetivo sólo
indica factibilidad. El caller registra tiempo y procedencia como costo docente.

La formulación sin índices de vehículo requiere que CADA capacidad pueda
transportar toda la demanda. Si no se cumple, devuelve unsupported_capacity;
no relaja ni agrega vehículos. AddMultipleCircuit exige que cada circuito pase
por el depósito, incluso cuando hay tiempos cero. Clientes de demanda cero son
obligatorios mediante self-loops falsos. Sobran vehículos: se conservan vacíos.

API primaria OR-Tools: https://or-tools.github.io/docs/pdoc/ortools/sat/python/cp_model.html
"""
from __future__ import annotations

from collections.abc import Mapping
import importlib.metadata
import math
from numbers import Integral, Real
import time


IMPLEMENTATION = 'project_independent_cp_sat_teacher_v1'
DEFAULTS = {'seconds': 600., 'workers': 1}
_TOL = 1e-9
_MAX_SAFE_INTEGER = 2**60


class _BudgetExpired(Exception):
    pass


def _deadline(deadline):
    if time.perf_counter() >= deadline:
        raise _BudgetExpired


def validate_config(config=None):
    if config is not None and not isinstance(config, Mapping):
        raise ValueError('La configuración del profesor CP-SAT debe ser un mapping.')
    unknown = set(config or {}) - set(DEFAULTS)
    if unknown:
        raise ValueError(f'Parámetros del profesor desconocidos: {sorted(unknown)}')
    result = DEFAULTS | dict(config or {})
    seconds = result['seconds']
    if (isinstance(seconds, bool) or not isinstance(seconds, Real)
            or not math.isfinite(float(seconds)) or seconds < 0):
        raise ValueError('seconds debe ser finito y no negativo.')
    workers = result['workers']
    if isinstance(workers, bool) or not isinstance(workers, Integral) or workers != 1:
        raise ValueError('El profesor reproducible requiere workers=1.')
    return {'seconds': float(seconds), 'workers': 1}


def _cp_sat():
    # Importar este módulo no carga OR-Tools ni inicia ningún solver.
    try:
        from ortools.sat.python import cp_model
    except ImportError as exc:
        raise RuntimeError('El profesor requiere OR-Tools instalado en Google Colab.') from exc
    return cp_model


def _integer_problem(problem, deadline):
    """Reconstruye la rejilla ya fijada; nunca redondea una matriz distinta."""
    _deadline(deadline)
    scale = problem.time_scale
    if isinstance(scale, bool) or not isinstance(scale, Integral) or scale < 1:
        raise ValueError('SearchProblem.time_scale debe ser entero positivo.')
    scale = int(scale)
    matrix = []
    for i in range(problem.n + 1):
        _deadline(deadline)
        row = []
        for j in range(problem.n + 1):
            value = float(problem.travel[i, j]) * scale
            if not math.isfinite(value) or value < 0 or value > _MAX_SAFE_INTEGER:
                raise ValueError('La matriz de búsqueda tiene tiempos inválidos.')
            integer = round(value)
            if not math.isclose(value, integer, rel_tol=0., abs_tol=1e-6):
                raise ValueError('La matriz de búsqueda no pertenece a su time_scale; no redondearla de nuevo.')
            if i == j and integer != 0:
                raise ValueError('La diagonal de viajes debe ser cero.')
            row.append(int(integer))
        matrix.append(row)
    lower, upper = [0], [0]
    horizon_value = float(problem.horizon) * scale
    if not math.isfinite(horizon_value) or not 0 <= horizon_value <= _MAX_SAFE_INTEGER:
        raise ValueError('Horizonte inválido para CP-SAT.')
    horizon = math.floor(horizon_value)
    upper[0] = horizon
    for customer in range(1, problem.n + 1):
        _deadline(deadline)
        start = float(problem.open[customer]) * scale
        end = float(problem.close[customer]) * scale
        if not all(math.isfinite(value) and 0 <= value <= _MAX_SAFE_INTEGER for value in (start, end)):
            raise ValueError('Ventanas inválidas para CP-SAT.')
        lower.append(math.ceil(start))
        upper.append(min(math.floor(end), horizon))
    return {'matrix': matrix, 'lower': lower, 'upper': upper, 'horizon': horizon, 'time_scale': scale}


def _build_model(problem, data, cp, deadline):
    model = cp.CpModel()
    times = {}
    for customer in range(1, problem.n + 1):
        _deadline(deadline)
        lo, hi = data['lower'][customer], data['upper'][customer]
        if lo > hi:
            model.add(False)
        times[customer] = model.new_int_var(min(lo, hi), hi, f'start_{customer}')
    edges, variables = [], {}
    # Registrar incluso clientes aislados. El literal falso impide omitirlos.
    for customer in range(1, problem.n + 1):
        edges.append((customer, customer, model.new_constant(0)))
    departures = []
    for origin in range(problem.n + 1):
        _deadline(deadline)
        for target in range(problem.n + 1):
            if origin == target:
                continue
            transit = data['matrix'][origin][target]
            earliest = 0 if origin == 0 else data['lower'][origin]
            latest = data['horizon'] if target == 0 else data['upper'][target]
            if earliest + transit > latest:
                continue
            selected = model.new_bool_var(f'edge_{origin}_{target}')
            edges.append((origin, target, selected))
            variables[origin, target] = selected
            if origin == 0:
                departures.append(selected)
                model.add(times[target] >= transit).only_enforce_if(selected)
            elif target == 0:
                model.add(times[origin] + transit <= data['horizon']).only_enforce_if(selected)
            else:
                model.add(times[target] >= times[origin] + transit).only_enforce_if(selected)
    # No objective and no hints. The circuit constraint excludes subtours even
    # when all their arc durations are zero; time inequalities alone would not.
    model.add_multiple_circuit(edges)
    model.add(sum(departures) <= problem.vehicles)
    _deadline(deadline)
    return model, variables


def _extract_routes(problem, selected):
    departures = sorted(target for origin, target in selected if origin == 0)
    if len(departures) > problem.vehicles:
        raise ValueError('CP-SAT excedió la flota fija.')
    successors = {}
    for origin, target in selected:
        if origin:
            if origin in successors:
                raise ValueError('CP-SAT devolvió dos sucesores para un cliente.')
            successors[origin] = target
    seen, routes = set(), []
    for first in departures:
        route, customer = [], first
        while customer:
            if customer in seen or customer not in successors or not 1 <= customer <= problem.n:
                raise ValueError('La extracción encontró ciclo desconectado o cliente inválido.')
            seen.add(customer)
            route.append(customer)
            customer = successors[customer]
        routes.append(route)
    if seen != set(range(1, problem.n + 1)):
        raise ValueError('La solución CP-SAT no cubre todos los clientes.')
    return routes + [[] for _ in range(problem.vehicles - len(routes))]


def _finish(problem, routes, status, started, config, metadata, history):
    validation_started = time.perf_counter()
    result = problem.result(routes, status, validation_started - started, history, **metadata)
    finished = time.perf_counter()
    result['solve_seconds'] = finished - started
    result['metadata']['validation_seconds'] += finished - validation_started
    result['metadata']['budget_overrun_seconds'] = max(0., finished - started - config['seconds'])
    return result


def solve_feasibility(problem, seed, config=None):
    """Obtiene una etiqueta independiente, sin optimizar costo ni usar certificados.

    El presupuesto único incluye importación, preparación, modelo, solver,
    extracción y validación. No interrumpe la operación nativa en curso; deja
    constancia de cualquier exceso. UNKNOWN no prueba infactibilidad.
    """
    started = time.perf_counter()
    effective = validate_config(config)
    if isinstance(seed, bool) or not isinstance(seed, Integral) or seed < 0:
        raise ValueError('seed debe ser entero no negativo.')
    seed = int(seed)
    deadline = started + effective['seconds']
    metadata = {
        'method': 'cp_sat_feasibility', 'seed': seed, 'cp_sat_seed': seed % (2**31 - 1),
        'implementation': IMPLEMENTATION, 'effective_config': effective,
        'construction': 'independent_from_scratch_without_certificates_or_hints',
        'feasibility_only': True, 'surrogate_optimal': False, 'stochastic_optimal': False,
        'cost_optimization_performed': False, 'best_bound': None,
        'cp_sat_status': None, 'cp_sat_status_name': 'NOT_RUN', 'solver_version': None,
        'termination': 'not_started', 'search_solver_seconds': 0., 'model_seconds': 0.,
        'import_seconds': 0., 'validation_seconds': 0., 'response_stats': None,
        'time_scale': int(problem.time_scale), 'capacity_scope': 'nonbinding_for_each_vehicle',
        'conservative_integer_infeasible': False,
        'infeasibility_scope': 'conservative_integer_model_only_not_original_stochastic_problem',
        'optimal_status_scope': 'satisfaction_of_feasibility_constraints_not_minimum_route_cost',
        'budget_includes_imports': True, 'budget_includes_construction': True,
        'budget_includes_extraction_and_validation': True,
        'time_limit_granularity': 'model_row_or_native_solver_call_or_validation',
        'seed_reproducibility': 'same_ortools_environment_and_one_worker; wall_time_cutoff_may_differ',
    }
    routes, history, status = None, [], 'no_solution'
    try:
        _deadline(deadline)
        total_demand = math.fsum(float(value) for value in problem.demands[1:])
        if (not math.isfinite(total_demand) or total_demand < 0
                or any(not math.isfinite(float(value)) or float(value) < 0 for value in problem.demands)):
            raise ValueError('Demandas inválidas.')
        if any(not math.isfinite(float(capacity)) or float(capacity) < 0 for capacity in problem.capacities):
            raise ValueError('Capacidades inválidas.')
        if problem.vehicles < 1 or any(float(capacity) + _TOL < total_demand for capacity in problem.capacities):
            metadata['termination'] = 'unsupported_capacity'
            return _finish(problem, None, 'unsupported_capacity', started, effective, metadata, history)
        model_started = time.perf_counter()
        try:
            data = _integer_problem(problem, deadline)
        finally:
            metadata['model_seconds'] += time.perf_counter() - model_started
        if problem.n == 0:
            routes = [[] for _ in range(problem.vehicles)]
            metadata['termination'] = 'empty_instance_feasible'
            return _finish(problem, routes, 'feasible', started, effective, metadata, history)
        import_started = time.perf_counter()
        cp = _cp_sat()
        metadata['solver_version'] = importlib.metadata.version('ortools')
        metadata['import_seconds'] = time.perf_counter() - import_started
        _deadline(deadline)
        model_started = time.perf_counter()
        try:
            model, variables = _build_model(problem, data, cp, deadline)
            model_error = model.validate()
        finally:
            metadata['model_seconds'] += time.perf_counter() - model_started
        if model_error:
            metadata.update(termination='model_invalid', model_validation_error=model_error,
                            cp_sat_status_name='MODEL_INVALID')
            return _finish(problem, None, 'model_invalid', started, effective, metadata, history)
        solver = cp.CpSolver()
        solver.parameters.num_search_workers = effective['workers']
        solver.parameters.random_seed = metadata['cp_sat_seed']
        solver.parameters.stop_after_first_solution = True
        _deadline(deadline)
        solver.parameters.max_time_in_seconds = max(0., deadline - time.perf_counter())
        # A zero native limit can mean no limit in solver APIs: never invoke it.
        if solver.parameters.max_time_in_seconds <= 0:
            raise _BudgetExpired
        solver_started = time.perf_counter()
        result_code = solver.solve(model)
        metadata.update(search_solver_seconds=time.perf_counter() - solver_started,
                        cp_sat_status=int(result_code), cp_sat_status_name=solver.status_name(result_code),
                        response_stats=solver.response_stats(), model_arc_count=len(variables))
        if result_code in (cp.FEASIBLE, cp.OPTIMAL):
            validation_started = time.perf_counter()
            selected = [arc for arc, variable in variables.items() if solver.boolean_value(variable)]
            candidate = _extract_routes(problem, selected)
            violation, cost = problem.score(candidate)
            if not math.isfinite(violation) or not math.isfinite(cost) or violation > _TOL:
                raise ValueError('La etiqueta CP-SAT no pasó la validación conservadora común.')
            metadata['validation_seconds'] += time.perf_counter() - validation_started
            routes, status = candidate, 'feasible'
            metadata['termination'] = 'feasible_found'
            history.append({'phase': 'independent_feasibility_teacher',
                            'elapsed_seconds': time.perf_counter() - started,
                            'violation_minutes': float(violation), 'travel_minutes': float(cost)})
        elif result_code == cp.INFEASIBLE:
            metadata.update(termination='conservative_integer_infeasible', conservative_integer_infeasible=True)
        elif result_code == cp.MODEL_INVALID:
            status, metadata['termination'] = 'model_invalid', 'model_invalid'
        else:
            metadata['termination'] = 'unknown_no_feasible_solution'
    except _BudgetExpired:
        metadata['termination'] = 'time_limit_before_search'
    return _finish(problem, routes, status, started, effective, metadata, history)


Writing svrplab/teacher_feasibility.py


In [9]:
%%writefile svrplab/supervised_data.py
"""Corpus heurístico auditado y particiones por geometría para el método 03.

La construcción se ejecuta exclusivamente en Colab. ACO/Tabú y el respaldo
CP-SAT parten de cero: nunca leen certificados ni escenarios de evaluación.
Los cuatro folds del piloto son desarrollo cruzado, no un test confirmatorio.
"""
from __future__ import annotations

import math
import shutil
import time
from collections.abc import Mapping
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

from . import experiments as exp
from . import metaheuristics as mh
from . import pilot as p
from .search import OBJECTIVE, SearchProblem

SCHEMA = 'supervised-teachers-v2'
TEACHER_DEFAULTS = {
    'seeds': [101, 211, 307], 'seconds_per_method': 120., 'time_scale': 1000,
    'aco': {'max_iterations': 200, 'ants': 20, 'alpha': 1., 'beta': 2.,
            'evaporation': .1, 'q0': .2},
    'tabu': {'max_iterations': 200, 'neighborhood_samples': 80, 'tenure': 15,
             'initial_restarts': 20},
    'fallback': {'enabled': False, 'seeds': [401, 503, 607],
                 'seconds_per_attempt': 600., 'workers': 1},
}
_ROLES = ('train_geometry_groups', 'validation_geometry_groups',
          'heldout_geometry_groups')
_COSTS = ('teacher_search_seconds', 'teacher_preparation_seconds',
          'teacher_validation_seconds', 'teacher_wall_seconds')


def _require_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise RuntimeError('Experimentación exclusivamente en Google Colab.') from exc


def _registry(instances):
    rows = []
    for x in instances:
        group = x.provenance.get('geometry_group_sha256')
        if (not isinstance(group, str) or len(group) != 64
                or any(c not in '0123456789abcdef' for c in group)):
            raise ValueError('Cada instancia requiere geometry_group_sha256 válido.')
        rows.append({'instance_id': x.instance_id, 'instance_sha256': x.sha,
                     'geometry_group_sha256': group, 'n_customers': x.n})
    if not rows or len({r['instance_id'] for r in rows}) != len(rows):
        raise ValueError('Registro de instancias vacío o con identificadores repetidos.')
    if len({r['instance_sha256'] for r in rows}) != len(rows):
        raise ValueError('Registro de instancias con hashes repetidos.')
    return sorted(rows, key=lambda r: (r['geometry_group_sha256'], r['instance_sha256']))


def make_geometry_folds(instances):
    """Cada grupo es heldout una vez; validación es el siguiente hash circular.

    Con las cuatro geometrías del piloto quedan dos grupos de entrenamiento,
    uno de validación y uno excluido de ambos. Ningún derivado cambia de rol.
    """
    instances = list(instances)
    groups = sorted({r['geometry_group_sha256'] for r in _registry(instances)})
    if len(groups) < 3:
        raise ValueError('Se requieren al menos tres geometrías independientes para los folds.')
    folds = []
    for i, heldout in enumerate(groups):
        validation = groups[(i + 1) % len(groups)]
        fold = {'fold_id': f'geometry_{i:02d}',
                'train_geometry_groups': [g for g in groups if g not in (heldout, validation)],
                'validation_geometry_groups': [validation],
                'heldout_geometry_groups': [heldout]}
        validate_fold(fold, instances)
        folds.append(fold)
    return folds


def validate_fold(fold, instances):
    """Rechaza cualquier mezcla de roles, grupo ajeno u omisión del registro."""
    known = {r['geometry_group_sha256'] for r in _registry(instances)}
    if (not isinstance(fold, Mapping) or not isinstance(fold.get('fold_id'), str)
            or not fold['fold_id'] or Path(fold['fold_id']).name != fold['fold_id']
            or fold['fold_id'] in ('.', '..')):
        raise ValueError('fold_id inválido.')
    used = set()
    for role in _ROLES:
        values = fold.get(role)
        if (not isinstance(values, (list, tuple)) or not values
                or any(not isinstance(v, str) for v in values)
                or len(set(values)) != len(values)):
            raise ValueError(f'Rol vacío, repetido o inválido: {role}')
        group_set = set(values)
        if not group_set <= known or used & group_set:
            raise ValueError('Grupos desconocidos o solapados entre train/validation/heldout.')
        used.update(group_set)
    if used != known or len(fold['heldout_geometry_groups']) != 1:
        raise ValueError('Todos los grupos deben tener exactamente un rol y un solo heldout.')
    return True



def validate_holdout_fold(fold, instances):
    """Partición final explícita: varias geometrías test, ninguna reutilizada.

    El runner final verifica además la reserva sellada y aporta su hash. Este
    validador comprueba exhaustividad/disjunción sin relajar los folds históricos.
    """
    known = {r['geometry_group_sha256'] for r in _registry(instances)}
    if (not isinstance(fold, Mapping) or fold.get('split_strategy') != 'independent_train_validation_test_v1'
            or not isinstance(fold.get('fold_id'), str) or not fold['fold_id']
            or Path(fold['fold_id']).name != fold['fold_id'] or fold['fold_id'] in ('.', '..')):
        raise ValueError('Partición holdout final inválida.')
    _check_hash(fold.get('split_registry_sha256'), 'split_registry_sha256')
    used = set()
    for role in _ROLES:
        values = fold.get(role)
        if (not isinstance(values, list) or not values or len(set(values)) != len(values)
                or any(not isinstance(value, str) for value in values)):
            raise ValueError('Rol holdout vacío o repetido.')
        group_set = set(values)
        if not group_set <= known or used & group_set:
            raise ValueError('Geometría ajena o compartida entre train/validation/test.')
        used.update(group_set)
    if used != known:
        raise ValueError('Partición holdout omite geometrías de la población.')
    return True

def _configuration(config):
    if not isinstance(config, Mapping) or set(config) - set(TEACHER_DEFAULTS):
        raise ValueError('Configuración de profesores inválida o con campos desconocidos.')
    cfg = {**TEACHER_DEFAULTS, **dict(config)}
    seeds = cfg['seeds']
    if (not isinstance(seeds, (list, tuple)) or not seeds
            or any(type(s) is not int or s < 0 for s in seeds)
            or len(set(seeds)) != len(seeds)):
        raise ValueError('Semillas de profesores inválidas o repetidas.')
    seconds = cfg['seconds_per_method']
    if (type(seconds) not in (int, float) or not math.isfinite(seconds) or seconds <= 0
            or type(cfg['time_scale']) is not int or cfg['time_scale'] <= 0):
        raise ValueError('Presupuesto y escala temporal deben ser positivos y finitos.')
    cfg['seeds'] = list(seeds)
    cfg['seconds_per_method'] = float(seconds)
    for name, defaults in [('aco', mh.ACO_DEFAULTS), ('tabu', mh.TABU_DEFAULTS)]:
        if not isinstance(cfg[name], Mapping) or 'seconds' in cfg[name]:
            raise ValueError('Use seconds_per_method para el presupuesto común de profesores.')
        cfg[name] = mh._configuration(defaults, {**cfg[name], 'seconds': float(seconds)})
        cfg[name].pop('seconds')
    fallback = cfg['fallback']
    if (not isinstance(fallback, Mapping)
            or set(fallback) - set(TEACHER_DEFAULTS['fallback'])):
        raise ValueError('Configuración de respaldo inválida.')
    fallback = {**TEACHER_DEFAULTS['fallback'], **dict(fallback)}
    if type(fallback['enabled']) is not bool:
        raise ValueError('fallback.enabled debe ser booleano.')
    seeds = fallback['seeds']
    if (not isinstance(seeds, (list, tuple)) or not seeds
            or any(type(seed) is not int or seed < 0 for seed in seeds)
            or len(set(seeds)) != len(seeds)):
        raise ValueError('Semillas de respaldo inválidas.')
    seconds = fallback['seconds_per_attempt']
    if (type(seconds) not in (int, float) or not math.isfinite(seconds) or seconds <= 0
            or type(fallback['workers']) is not int or fallback['workers'] != 1):
        raise ValueError('Respaldo requiere presupuesto positivo y un worker.')
    cfg['fallback'] = {**fallback, 'seeds': list(seeds), 'seconds_per_attempt': float(seconds)}
    return cfg


def solve_feasibility(problem, seed, config):
    """Dependencia diferida: los folds de 04/05 no necesitan cargar un solver."""
    from .teacher_feasibility import solve_feasibility as solve
    return solve(problem, seed, config)


def _check_hash(value, name):
    if (not isinstance(value, str) or len(value) != 64
            or any(c not in '0123456789abcdef' for c in value)):
        raise ValueError(f'Hash inválido: {name}')


def _check_result(problem, result, spec):
    """Validez de etiqueta: estructura, cota conservadora y simulador nominal."""
    validation = {'eligible_label': False, 'structural_valid': False,
                  'conservative_feasible': False, 'nominal_feasible': False,
                  'reason': 'no_solution', 'objective_scope': OBJECTIVE}
    nominal = None
    try:
        routes = result.get('routes')
        if routes is None:
            return validation, nominal
        p.validate_routes(problem.x, routes)
        validation['structural_valid'] = True
        violation, cost = problem.score(routes)
        if not math.isfinite(violation) or not math.isfinite(cost):
            raise ValueError('Puntuación de profesor no finita.')
        validation.update(search_violation_minutes=float(violation),
                          search_objective_minutes=float(cost),
                          conservative_feasible=violation <= 1e-9)
        nominal_spec = p.Traffic(**(asdict(spec) | {'mode': 'nominal_mean'}))
        # El 0 es una identidad formal: nominal_mean no muestrea ruido.
        nominal = p.simulate(problem.x, routes, 0, nominal_spec, 'fixed_order', trace=True)
        validation['nominal_feasible'] = bool(nominal['strict_feasible'])
        validation['eligible_label'] = validation['conservative_feasible'] and validation['nominal_feasible']
        validation['reason'] = ('accepted_heuristic_label' if validation['eligible_label'] else
                                'conservative_infeasible' if not validation['conservative_feasible'] else
                                'nominal_infeasible')
    except Exception as exc:
        validation.update(reason='validation_error', error_type=type(exc).__name__, error=str(exc))
    return validation, nominal


def _run_attempt(x, spec, cfg, method, seed, identity, directory):
    """Persiste incluso soluciones rechazadas; retorna KeyboardInterrupt diferido."""
    directory.mkdir(parents=True)
    started = time.perf_counter()
    preparation = search_seconds = validation_seconds = 0.
    result = {'routes': None, 'history': [], 'metadata': {}, 'status': 'solver_error',
              'solve_seconds': 0.}
    validation = {'eligible_label': False, 'structural_valid': False,
                  'conservative_feasible': False, 'nominal_feasible': False,
                  'reason': 'solver_error'}
    nominal, interrupt = None, None
    problem, search_started = None, None
    try:
        problem = SearchProblem(x, spec, cfg['time_scale'])
        preparation = time.perf_counter() - started
        search_started = time.perf_counter()
        budget = max(0., identity['search_budget_seconds'] - preparation)
        if method == 'cp_sat_feasibility':
            solver = solve_feasibility
            settings = {'workers': cfg['fallback']['workers'], 'seconds': budget}
        else:
            solver = {'aco': mh.solve_aco, 'tabu': mh.solve_tabu}[method]
            settings = {**cfg[method], 'seconds': budget}
        result = solver(problem, seed, settings)
        if not isinstance(result, dict):
            raise TypeError('El profesor debe devolver un diccionario de resultado.')
        search_seconds = time.perf_counter() - search_started
        search_started = None
        validation_started = time.perf_counter()
        validation, nominal = _check_result(problem, result, spec)
        validation_seconds = time.perf_counter() - validation_started
    except (Exception, KeyboardInterrupt) as exc:
        if search_started is not None:
            search_seconds = time.perf_counter() - search_started
        elif problem is None:
            preparation = time.perf_counter() - started
        if not isinstance(result, dict):
            result = {'routes': None, 'history': [], 'metadata': {}}
        result.update(status='interrupted' if isinstance(exc, KeyboardInterrupt) else 'solver_error')
        result['metadata'] = {**result.get('metadata', {}),
                              'error_type': type(exc).__name__, 'error': str(exc)}
        validation.update(eligible_label=False, reason=result['status'])
        if isinstance(exc, KeyboardInterrupt):
            interrupt = exc
    if x.sha != identity['instance_sha256']:
        raise AssertionError('El profesor modificó la instancia.')
    result.setdefault('history', [])
    result.setdefault('metadata', {})
    result.setdefault('routes', None)
    result.setdefault('solve_seconds', search_seconds)
    # JSON conserva rutas de intentos inválidos; el CSV sólo requiere estructura válida.
    exp.write_json(directory / 'result.json', result)
    exp.write_json(directory / 'nominal.json', nominal)
    exp.write_json(directory / 'validation.json', validation)
    exp.write_csv(directory / 'search_history.csv', result['history'],
                  None if result['history'] else ['iteration', 'elapsed_seconds'])
    rows = exp.route_rows(x, result['routes'], identity) if validation['structural_valid'] else []
    exp.write_csv(directory / 'planned_routes.csv', rows,
                  None if rows else ['instance_id', 'teacher_method', 'teacher_seed', 'vehicle', 'position', 'node'])
    if nominal is not None:
        exp.write_csv(directory / 'nominal_events.csv', nominal['events'])
    costs = {'teacher_preparation_seconds': float(preparation),
             'teacher_search_seconds': float(search_seconds),
             'teacher_validation_seconds': float(validation_seconds),
             'teacher_wall_seconds': time.perf_counter() - started,
             'wall_scope': 'preparation_search_validation_local_artifacts_before_final_record; excludes_drive_sync'}
    exp.write_json(directory / 'attempt.json', {**identity, **costs,
                   'status': result['status'], 'label_validation': validation,
                   'solver_termination': result['metadata'].get('termination'),
                   'solver_iterations': result['metadata'].get('iterations'),
                   'solver_error_type': result['metadata'].get('error_type'),
                   'solver_error': result['metadata'].get('error'),
                   'routes_sha256': p.digest(result['routes']) if result['routes'] is not None else None,
                   'construction': 'independent_from_scratch_without_certificates',
                   'label_kind': 'heuristic_not_proven_optimal'})
    return interrupt


def _cost_summary(attempts, groups):
    costs = []
    for group in sorted(groups):
        subset = [a for a in attempts if a['geometry_group_sha256'] == group]
        costs.append({'geometry_group_sha256': group, 'attempts': len(subset),
                      'accepted_attempts': sum(a['label_validation']['eligible_label'] for a in subset),
                      'rejected_attempts': sum(not a['label_validation']['eligible_label'] for a in subset),
                      **{key: sum(a[key] for a in subset) for key in _COSTS}})
    return costs


def build_teacher_corpus(instances, spec, config, work_root, drive_root, bank_id,
                         bank_sha256, evaluator_sha256):
    """Genera o recupera cada intento inmutable y retorna la raíz del corpus.

    El llamador debe cargar antes el banco aceptado mediante p.load_bank.
    Identidad: banco, registro íntegro, tráfico, código y configuración efectiva.
    Sólo se sella index/ al terminar: nunca la raíz que recibe nuevos intentos.
    """
    _require_colab()
    instances = list(instances)
    registry, cfg = _registry(instances), _configuration(config)
    folds = make_geometry_folds(instances)
    if not isinstance(bank_id, str) or Path(bank_id).name != bank_id or bank_id in ('', '.', '..'):
        raise ValueError('bank_id debe ser un nombre de carpeta simple.')
    _check_hash(bank_sha256, 'bank_sha256')
    _check_hash(evaluator_sha256, 'evaluator_sha256')
    if p.file_hash(p.__file__) != evaluator_sha256:
        raise ValueError('El evaluador activo no coincide con el banco sellado.')
    package = Path(__file__).parent
    source_hashes = {name: p.file_hash(package / name) for name in
                     ('supervised_data.py', 'metaheuristics.py', 'baselines.py', 'search.py', 'experiments.py')}
    source_hashes['pilot.py'] = evaluator_sha256
    if cfg['fallback']['enabled']:
        source_hashes['teacher_feasibility.py'] = p.file_hash(package / 'teacher_feasibility.py')
    protocol = {'schema_version': SCHEMA, 'role': 'development_geometry_crossfit',
                'bank_id': bank_id, 'bank_manifest_sha256': bank_sha256,
                'evaluator_sha256': evaluator_sha256, 'instances': registry,
                'traffic': asdict(spec), 'config': cfg, 'source_hashes': source_hashes,
                'objective_scope': OBJECTIVE, 'folds': folds,
                'labels': 'all_distinct_eligible_heuristic_final_incumbents',
                'fallback_trigger': 'zero_eligible_base_labels_for_instance_after_all_aco_tabu_attempts',
                'fallback_seed_policy': 'all_predeclared_seeds_even_after_success',
                'fallback_objective': 'feasibility_only_not_cost_optimality',
                'independent_unit': 'geometry_group_sha256'}
    corpus_id = p.digest(protocol)
    root = Path(work_root) / 'teacher_corpora' / bank_id / corpus_id
    remote = Path(drive_root) / 'teacher_corpora' / bank_id / corpus_id
    if root.resolve() == remote.resolve():
        raise ValueError('Trabajo local y Drive requieren directorios distintos.')
    print(f'TEACHER_CORPUS_ID = {corpus_id}\nDrive: {remote}', flush=True)
    if exp._recover(root / 'meta', remote / 'meta'):
        if exp.read_json(root / 'meta/protocol.json') != protocol:
            raise ValueError('Protocolo de profesores distinto del checkpoint.')
    else:
        exp.write_json(root / 'meta/protocol.json', protocol)
        exp.write_json(root / 'meta/identity.json', {'corpus_id': corpus_id,
                       'created_utc': datetime.now(timezone.utc).isoformat()})
        exp.write_json(root / 'meta/environment.json', p.environment_report())
        code_dir = root / 'meta/code'
        code_dir.mkdir()
        for name in source_hashes:
            shutil.copy2(Path(p.__file__) if name == 'pilot.py' else package / name, code_dir / name)
        exp._commit(root / 'meta', remote / 'meta')
    attempts, labels, seen, inventory = [], [], set(), {'meta': p.verify_bundle(root / 'meta')}
    by_sha = {x.sha: x for x in instances}
    def collect(row, method, seed, budget):
        x = by_sha[row['instance_sha256']]
        key = f'attempts/{x.sha}/{method}/seed_{seed}'
        identity = {**row, 'corpus_id': corpus_id, 'bank_id': bank_id,
                    'bank_manifest_sha256': bank_sha256,
                    'evaluator_sha256': evaluator_sha256,
                    'teacher_method': method, 'teacher_seed': seed,
                    'search_budget_seconds': budget, 'attempt_key': key}
        directory = root / key
        recovered = exp._recover(directory, remote / key)
        if not recovered:
            print(f'Profesor {method} · {x.instance_id} · seed={seed} · presupuesto={budget:g}s', flush=True)
            interrupt = _run_attempt(x, spec, cfg, method, seed, identity, directory)
            exp._commit(directory, remote / key)
            if interrupt is not None:
                raise interrupt
        attempt = exp.read_json(directory / 'attempt.json')
        if any(attempt.get(k) != v for k, v in identity.items()):
            raise ValueError('Identidad de intento incompatible con el corpus.')
        attempts.append(attempt)
        inventory[key] = p.verify_bundle(directory)
        if p.verify_bundle(remote / key) != inventory[key]:
            raise ValueError('Intento en Drive distinto del local.')
        validation = attempt['label_validation']
        print(f"  {'recuperado' if recovered else 'terminado'} · n={x.n} · {method} · seed={seed} · "
              f"estado={attempt['status']} · {validation['reason']} · "
              f"violación={validation.get('search_violation_minutes')} · "
              f"búsqueda={attempt['teacher_search_seconds']:.2f}s · "
              f"terminación={attempt.get('solver_termination')}", flush=True)
        if attempt.get('solver_error') or validation.get('error'):
            print('  error:', attempt.get('solver_error') or validation.get('error'), flush=True)
        label_key = (x.sha, attempt['routes_sha256'])
        if validation['eligible_label'] and label_key not in seen:
            seen.add(label_key)
            labels.append({**row, 'routes_sha256': attempt['routes_sha256'],
                           'attempt_key': key, 'teacher_method': method, 'teacher_seed': seed,
                           'objective_minutes': validation['search_objective_minutes']})

    for row in registry:
        for method in ('aco', 'tabu'):
            for seed in cfg['seeds']:
                collect(row, method, seed, cfg['seconds_per_method'])
        if cfg['fallback']['enabled'] and not any(
                label['instance_sha256'] == row['instance_sha256'] for label in labels):
            print(f"Respaldo independiente CP-SAT · n={row['n_customers']} · "
                  'sin etiquetas ACO/Tabú; no se leen certificados.', flush=True)
            for seed in cfg['fallback']['seeds']:
                collect(row, 'cp_sat_feasibility', seed, cfg['fallback']['seconds_per_attempt'])
    groups = {r['geometry_group_sha256'] for r in registry}
    labelled_instances = {label['instance_sha256'] for label in labels}
    missing_instances = [r for r in registry if r['instance_sha256'] not in labelled_instances]
    missing = sorted({r['geometry_group_sha256'] for r in missing_instances})
    costs = _cost_summary(attempts, groups)
    index = {'corpus_id': corpus_id, 'schema_version': SCHEMA,
             'status': 'no_labels_for_geometry' if missing else 'complete',
             'missing_geometry_groups': missing, 'protocol_sha256': p.digest(protocol),
             'missing_instance_sha256': [r['instance_sha256'] for r in missing_instances],
             'instances': registry, 'labels': labels, 'attempts': len(attempts),
             'teacher_costs_by_geometry': costs, 'bundles': inventory,
             'label_kind': 'heuristic_not_proven_optimal',
             'ready_for_confirmatory_comparison': False}
    if exp._recover(root / 'index', remote / 'index'):
        if exp.read_json(root / 'index/corpus.json') != index:
            raise ValueError('Índice de corpus distinto de los intentos recuperados.')
    else:
        exp.write_json(root / 'index/corpus.json', index)
        # Una fila por intento, incluidos errores, duplicados y rutas no elegibles.
        export_rows = [{**{k: v for k, v in a.items() if k != 'label_validation'},
                        **a['label_validation']} for a in attempts]
        exp.write_csv(root / 'index/teacher_attempts.csv', export_rows)
        exp.write_csv(root / 'index/teacher_costs_by_geometry.csv', costs)
        exp.write_csv(root / 'index/labels.csv', labels,
                      None if labels else ['instance_sha256', 'geometry_group_sha256', 'attempt_key'])
        exp.write_json(root / 'index/folds.json', folds)
        exp._commit(root / 'index', remote / 'index')
    if _registry(instances) != registry:
        raise AssertionError('El registro del banco cambió durante los profesores.')
    if missing:
        raise RuntimeError('Corpus sin etiquetas válidas para geometrías: ' + ', '.join(missing)
                           + '. Instancias: ' + ', '.join(str(r['n_customers']) for r in missing_instances)
                           + f'. Intentos y costos conservados en {remote}. '
                             'Revisar index/teacher_attempts.csv (reason, solver_termination, solver_error). '
                             'No se remuestrea ni se reutilizan certificados.')
    return root


def load_fold_examples(corpus_dir, instances, fold):
    """Retorna (train, validation, provenance), sin devolver rutas heldout.

    La verificación de integridad contempla todos los paquetes, pero sólo se
    abren result.json de grupos elegibles para entrenamiento o validación.
    """
    root = Path(corpus_dir)
    instances = list(instances)
    validate_fold(fold, instances)
    p.verify_bundle(root / 'index')
    meta_hash = p.verify_bundle(root / 'meta')
    index = exp.read_json(root / 'index/corpus.json')
    protocol = exp.read_json(root / 'meta/protocol.json')
    registry = _registry(instances)
    if (index.get('status') != 'complete' or index.get('instances') != registry
            or protocol.get('instances') != registry
            or index.get('corpus_id') != p.digest(protocol)
            or index.get('protocol_sha256') != p.digest(protocol)
            or index.get('bundles', {}).get('meta') != meta_hash
            or fold not in protocol.get('folds', [])):
        raise ValueError('Corpus incompleto, registro incompatible o fold ajeno al protocolo.')
    if protocol['evaluator_sha256'] != p.file_hash(p.__file__):
        raise ValueError('Evaluador del corpus distinto del módulo activo.')
    for key, expected in index['bundles'].items():
        relative = Path(key)
        if relative.is_absolute() or '..' in relative.parts:
            raise ValueError('Ruta de paquete inválida en corpus.')
        if p.verify_bundle(root / relative) != expected:
            raise ValueError('Paquete de profesores modificado.')
    by_sha = {x.sha: x for x in instances}
    groups = {'train': set(fold['train_geometry_groups']),
              'validation': set(fold['validation_geometry_groups'])}
    examples = {'train': [], 'validation': []}
    for label in index['labels']:
        role = next((role for role, selected in groups.items()
                     if label['geometry_group_sha256'] in selected), None)
        if role is None:
            continue
        key = label['attempt_key']
        if key not in index['bundles']:
            raise ValueError('Etiqueta sin intento sellado.')
        attempt = exp.read_json(root / key / 'attempt.json')
        result = exp.read_json(root / key / 'result.json')
        x = by_sha.get(label['instance_sha256'])
        if (x is None or label['instance_id'] != x.instance_id
                or label['geometry_group_sha256'] != x.provenance['geometry_group_sha256']
                or any(attempt.get(k) != label[k] for k in
                       ('instance_sha256', 'geometry_group_sha256', 'attempt_key',
                        'teacher_method', 'teacher_seed', 'routes_sha256'))
                or attempt['label_validation']['eligible_label'] is not True
                or p.digest(result['routes']) != label['routes_sha256']):
            raise ValueError('Identidad o validación de etiqueta incompatible.')
        p.validate_routes(x, result['routes'])
        examples[role].append({'instance_id': x.instance_id, 'instance_sha256': x.sha,
                               'geometry_group_sha256': label['geometry_group_sha256'],
                               'routes': result['routes'], 'teacher': {
                                   'method': label['teacher_method'], 'seed': label['teacher_seed'],
                                   'attempt_key': key, 'objective_minutes': label['objective_minutes'],
                                   'label_kind': 'heuristic_not_proven_optimal'}})
    for role in groups:
        if {e['geometry_group_sha256'] for e in examples[role]} != groups[role]:
            raise ValueError(f'Faltan etiquetas de una geometría del rol {role}.')
        expected_instances = {x.sha for x in instances
                              if x.provenance['geometry_group_sha256'] in groups[role]}
        if {e['instance_sha256'] for e in examples[role]} != expected_instances:
            raise ValueError(f'Faltan etiquetas de una instancia del rol {role}.')
    costs = index['teacher_costs_by_geometry']
    selected = groups['train'] | groups['validation']
    provenance = {'corpus_id': index['corpus_id'], 'fold_id': fold['fold_id'],
                  'bank_id': protocol['bank_id'],
                  'bank_manifest_sha256': protocol['bank_manifest_sha256'],
                  'evaluator_sha256': protocol['evaluator_sha256'],
                  'teacher_costs_by_geometry': costs,
                  'selected_teacher_costs': {key: sum(c[key] for c in costs
                                                     if c['geometry_group_sha256'] in selected)
                                             for key in _COSTS},
                  **{f'{role}_teacher_costs': {
                      key: sum(c[key] for c in costs if c['geometry_group_sha256'] in group_set)
                      for key in _COSTS} for role, group_set in groups.items()},
                  'all_teacher_costs': {key: sum(c[key] for c in costs) for key in _COSTS},
                  'train_examples': len(examples['train']),
                  'validation_examples': len(examples['validation']),
                  'heldout_labels_exposed': False,
                  'independent_unit': 'geometry_group_sha256',
                  'role': 'development_geometry_crossfit'}
    return examples['train'], examples['validation'], provenance


Writing svrplab/supervised_data.py


In [10]:
%%writefile svrplab/supervised.py
"""03: imitación supervisada por geometría, checkpoints y adaptador del runner.

Sólo Colab. Ningún escenario evaluation interviene en etiquetas, gradientes o
selección de checkpoint. El piloto no es una prueba final de generalización.
"""
from __future__ import annotations

import math
import random
import time
import uuid
from dataclasses import asdict
from pathlib import Path

import numpy as np
import torch

from . import experiments as e
from . import pilot as p
from .attention import AttentionPolicy, teacher_loss, solve_attention
from .search import SearchProblem
from .supervised_data import make_geometry_folds, validate_fold, load_fold_examples


def require_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise RuntimeError('Entrenamiento y experimentación exclusivamente en Google Colab.') from exc


def validate_training_config(config):
    if (config['schema_version'] != 'supervised-03-v1' or config['role'] != 'development_only'
            or config['split_strategy'] != 'leave_one_geometry_out_with_inner_validation'):
        raise ValueError('Configuración supervisada incompatible.')
    settings = config['training']
    for name in ['epochs', 'steps_per_epoch', 'gradient_accumulation', 'patience']:
        if type(settings[name]) is not int or settings[name] < 1:
            raise ValueError(f'{name} debe ser entero positivo.')
    for name in ['learning_rate', 'gradient_clip_norm', 'seconds_per_model']:
        if isinstance(settings[name], bool) or not math.isfinite(settings[name]) or settings[name] <= 0:
            raise ValueError(f'{name} debe ser finito y positivo.')
    for name in ['weight_decay', 'min_delta']:
        if isinstance(settings[name], bool) or not math.isfinite(settings[name]) or settings[name] < 0:
            raise ValueError(f'{name} debe ser finito y no negativo.')
    for name in ['permute_nodes', 'deterministic_algorithms']:
        if type(settings[name]) is not bool:
            raise ValueError(f'{name} debe ser booleano.')
    return True


def synchronize(device):
    if torch.device(device).type == 'cuda':
        torch.cuda.synchronize(device)


def choose_device(requested='auto'):
    if requested == 'auto':
        requested = 'cuda' if torch.cuda.is_available() else 'cpu'
    device = torch.device(requested)
    if device.type not in ('cpu', 'cuda') or (device.type == 'cuda' and not torch.cuda.is_available()):
        raise ValueError('Dispositivo no disponible: usar cpu, cuda o auto.')
    return device


def _source_hashes():
    directory = Path(__file__).parent
    return {name: p.file_hash(directory / name) for name in
            ('supervised.py', 'supervised_data.py', 'attention.py', 'search.py', 'metaheuristics.py', 'baselines.py')}


def _group_examples(examples, allowed):
    groups = {group: [] for group in allowed}
    for example in examples:
        group = example['geometry_group_sha256']
        if group not in groups:
            raise ValueError('Una etiqueta pertenece a una geometría no autorizada para este split.')
        groups[group].append(example)
    if not groups or any(not values for values in groups.values()):
        raise ValueError('Faltan etiquetas para alguna geometría del split.')
    return groups


def validation_nll(model, problems, examples, groups, device):
    """Promedio de NLL por decisión, luego etiqueta y geometría; sin heldout."""
    grouped = _group_examples(examples, groups)
    model.eval()
    losses = {}
    with torch.no_grad():
        for group, items in grouped.items():
            values = []
            for example in items:
                loss, _ = teacher_loss(model, problems[example['instance_sha256']],
                                       example['routes'], device)
                value = float(loss.detach().cpu())
                if not math.isfinite(value):
                    raise ValueError('NLL de validación no finita.')
                values.append(value)
            losses[group] = float(np.mean(values))
    return float(np.mean(list(losses.values()))), losses


def _cpu_state(model):
    return {key: tensor.detach().cpu().clone() for key, tensor in model.state_dict().items()}


def _load_checkpoint(path):
    # Sólo bundles propios, previamente verificados; evita pickle arbitrario.
    return torch.load(path, map_location='cpu', weights_only=True)


def _runtime_signature(device):
    return {'torch': torch.__version__, 'numpy': np.__version__, 'device_type': torch.device(device).type,
            'cuda': torch.version.cuda,
            'device_name': torch.cuda.get_device_name(device) if torch.device(device).type == 'cuda' else 'cpu'}


def train_fold(instances, spec, config, seed, fold, corpus, directory, remote, device):
    require_colab()
    validate_training_config(config)
    if type(seed) is not int or seed < 0:
        raise ValueError('La semilla de entrenamiento debe ser un entero no negativo.')
    validate_fold(fold, instances)
    directory, remote = Path(directory), Path(remote)
    device = choose_device(str(device))
    started = time.perf_counter()
    train, validation, provenance = load_fold_examples(corpus, instances, fold)
    grouped = _group_examples(train, fold['train_geometry_groups'])
    _group_examples(validation, fold['validation_geometry_groups'])
    identity = {'schema_version': config['schema_version'], 'fold': fold, 'seed': seed,
                'config': config, 'source_hashes': _source_hashes(),
                'corpus_id': provenance['corpus_id'], 'traffic': asdict(spec),
                'train_examples_sha256': p.digest(train), 'validation_examples_sha256': p.digest(validation)}
    if e._recover(directory / 'meta', remote / 'meta'):
        if e.read_json(directory / 'meta/identity.json') != identity:
            raise ValueError('Checkpoint de entrenamiento incompatible con corpus, código, fold o parámetros.')
    else:
        e.write_json(directory / 'meta/identity.json', identity)
        e.write_json(directory / 'meta/provenance.json', provenance)
        e.write_json(directory / 'meta/environment.json', p.environment_report())
        e._commit(directory / 'meta', remote / 'meta')
    if e._recover(directory / 'final', remote / 'final'):
        if e.read_json(directory / 'final/summary.json')['identity_sha256'] != p.digest(identity):
            raise ValueError('Modelo final de otro entrenamiento.')
        return directory / 'final'
    instances_by_sha = {x.sha: x for x in instances}
    for role, examples in [('train_geometry_groups', train), ('validation_geometry_groups', validation)]:
        for example in examples:
            instance = instances_by_sha.get(example['instance_sha256'])
            if (instance is None or instance.provenance['geometry_group_sha256'] != example['geometry_group_sha256']
                    or instance.provenance['geometry_group_sha256'] not in fold[role]):
                raise ValueError('Etiqueta fuera del registro o del rol de geometría autorizado.')
    selected = {ex['instance_sha256'] for ex in train + validation}
    problems = {sha: SearchProblem(instances_by_sha[sha], spec, config['teacher']['time_scale']) for sha in selected}
    for example in train + validation:
        if problems[example['instance_sha256']].score(example['routes'])[0] > 1e-9:
            raise ValueError('Etiqueta sin factibilidad conservadora.')
    settings = config['training']
    derived_seed = p.seed_for('supervised-03', fold['fold_id'], seed)
    random.seed(derived_seed)
    torch.manual_seed(derived_seed)
    if device.type == 'cuda':
        torch.cuda.manual_seed_all(derived_seed)
    torch.use_deterministic_algorithms(settings['deterministic_algorithms'])
    torch.backends.cudnn.benchmark = False
    rng = np.random.default_rng(derived_seed)
    model = AttentionPolicy(config['model']).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=settings['learning_rate'],
                                 weight_decay=settings['weight_decay'])
    best_state, best_loss, best_epoch, stale = None, math.inf, 0, 0
    history, elapsed_total, validation_total, last_epoch, terminal = [], 0.0, 0.0, 0, None
    checkpoint_numbers = set()
    for location in (directory, remote):
        if location.exists():
            checkpoint_numbers.update(int(child.name[6:]) for child in location.glob('epoch_*')
                                      if child.name[6:].isdigit() and (child / 'CHECKSUMS.json').exists())
    if checkpoint_numbers and sorted(checkpoint_numbers) != list(range(1, max(checkpoint_numbers) + 1)):
        raise ValueError('Faltan épocas selladas en la secuencia de checkpoints.')
    for epoch in sorted(checkpoint_numbers):
        if not e._recover(directory / f'epoch_{epoch:04d}', remote / f'epoch_{epoch:04d}'):
            raise ValueError('Checkpoint de época no recuperable.')
    if checkpoint_numbers:
        last_epoch = max(checkpoint_numbers)
        latest = directory / f'epoch_{last_epoch:04d}'
        progress = e.read_json(latest / 'progress.json')
        if (progress['identity_sha256'] != p.digest(identity)
                or progress['runtime_signature'] != _runtime_signature(device)):
            raise ValueError('No se reanuda entrenamiento parcial con otro entorno/fold. Use el entorno original o una configuración nueva.')
        state = _load_checkpoint(latest / 'state.pt')
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        best_state = state['best_model']
        best_loss, best_epoch, stale = progress['best_validation_nll'], progress['best_epoch'], progress['stale_epochs']
        history = progress['history']
        elapsed_total, validation_total = progress['training_seconds'], progress['validation_seconds']
        rng.bit_generator.state = progress['numpy_generator_state']
        random.setstate((progress['python_rng_state'][0], tuple(progress['python_rng_state'][1]), progress['python_rng_state'][2]))
        torch.set_rng_state(state['torch_rng'])
        if device.type == 'cuda':
            torch.cuda.set_rng_state_all(state['cuda_rng'])
        terminal = progress['stop_reason']
    synchronize(device)
    elapsed_total += time.perf_counter() - started
    groups = sorted(grouped)
    for epoch in range(last_epoch + 1, settings['epochs'] + 1):
        if terminal:
            break
        if elapsed_total >= settings['seconds_per_model']:
            terminal = 'time_limit'
            break
        epoch_started = time.perf_counter()
        model.train()
        values, updates, decisions = [], 0, 0
        for offset in range(0, settings['steps_per_epoch'], settings['gradient_accumulation']):
            if elapsed_total + time.perf_counter() - epoch_started >= settings['seconds_per_model']:
                break
            batch_size = min(settings['gradient_accumulation'], settings['steps_per_epoch'] - offset)
            optimizer.zero_grad(set_to_none=True)
            for _ in range(batch_size):
                group = groups[int(rng.integers(len(groups)))]
                items = grouped[group]
                example = items[int(rng.integers(len(items)))]
                problem = problems[example['instance_sha256']]
                order = rng.permutation(np.arange(1, problem.n + 1)).tolist() if settings['permute_nodes'] else None
                loss, stats = teacher_loss(model, problem, example['routes'], device, node_order=order)
                if not bool(torch.isfinite(loss).item()):
                    raise ValueError('Pérdida de entrenamiento no finita.')
                (loss / batch_size).backward()
                values.append(float(loss.detach().cpu()))
                decisions += stats['decisions']
            norm = torch.nn.utils.clip_grad_norm_(model.parameters(), settings['gradient_clip_norm'], error_if_nonfinite=True)
            if not math.isfinite(float(norm)):
                raise ValueError('Gradientes no finitos.')
            optimizer.step(); updates += 1
        if not values:
            elapsed_total += time.perf_counter() - epoch_started
            terminal = 'time_limit'
            break
        synchronize(device)
        val_started = time.perf_counter()
        val_loss, group_losses = validation_nll(model, problems, validation, fold['validation_geometry_groups'], device)
        synchronize(device)
        val_seconds = time.perf_counter() - val_started
        if val_loss < best_loss - settings['min_delta']:
            best_loss, best_epoch, best_state, stale = val_loss, epoch, _cpu_state(model), 0
        else:
            stale += 1
        elapsed_total += time.perf_counter() - epoch_started
        validation_total += val_seconds
        terminal = ('patience' if stale >= settings['patience'] else
                    'time_limit' if elapsed_total >= settings['seconds_per_model'] else
                    'epochs' if epoch == settings['epochs'] else None)
        history.append({'epoch': epoch, 'train_nll': float(np.mean(values)), 'validation_nll': val_loss,
                        'best_validation_nll': best_loss, 'best_epoch': best_epoch,
                        'examples_sampled': len(values), 'decisions': decisions, 'optimizer_updates': updates,
                        'training_seconds': elapsed_total, 'validation_seconds': validation_total})
        target = directory / f'epoch_{epoch:04d}'
        # Preservar una escritura interrumpida de esta época antes de repetirla.
        if e._recover(target, remote / target.name):
            raise ValueError('La época nueva ya existe; no sobrescribir un checkpoint.')
        target.mkdir(parents=True, exist_ok=True)
        torch.save({'model': _cpu_state(model), 'optimizer': optimizer.state_dict(), 'best_model': best_state,
                    'torch_rng': torch.get_rng_state(),
                    'cuda_rng': torch.cuda.get_rng_state_all() if device.type == 'cuda' else []}, target / 'state.pt')
        e.write_json(target / 'progress.json', {'epoch': epoch, 'identity_sha256': p.digest(identity),
                     'runtime_signature': _runtime_signature(device), 'best_validation_nll': best_loss,
                     'best_epoch': best_epoch, 'stale_epochs': stale, 'history': history,
                     'training_seconds': elapsed_total, 'validation_seconds': validation_total,
                     'numpy_generator_state': rng.bit_generator.state, 'python_rng_state': random.getstate(),
                     'validation_by_geometry': group_losses, 'stop_reason': terminal})
        e.write_json(target / 'environment.json', p.environment_report())
        e._commit(target, remote / target.name)
        print(f'{fold["fold_id"]} seed={seed} epoch={epoch}: train={history[-1]["train_nll"]:.5f}, '
              f'val={val_loss:.5f}, best={best_epoch}', flush=True)
    if best_state is None:
        raise RuntimeError('No hay checkpoint entrenado y seleccionado; no evaluar pesos aleatorios.')
    final = directory / 'final'
    final.mkdir(parents=True, exist_ok=True)
    torch.save(best_state, final / 'best.pt')
    summary = {'identity_sha256': p.digest(identity), 'fold': fold, 'training_seed': seed,
               'training_seconds': elapsed_total, 'validation_seconds': validation_total,
               'training_budget_seconds': settings['seconds_per_model'],
               'training_budget_overrun_seconds': max(0., elapsed_total - settings['seconds_per_model']),
               'training_time_scope': 'setup_and_epoch_compute_validation_excludes_epoch_and_final_checkpoint_persistence',
               'teacher_seconds': provenance['selected_teacher_costs']['teacher_wall_seconds'],
               'best_epoch': best_epoch, 'best_validation_nll': best_loss, 'epochs_completed': len(history),
               'stop_reason': terminal, 'model_config': config['model'], 'traffic': asdict(spec),
               'corpus_id': provenance['corpus_id'], 'checkpoint_selection': 'inner_validation_teacher_nll',
               'training_scope': 'development_geometry_crossfit_surrogate_imitation',
               'train_examples': len(train), 'validation_examples': len(validation),
               'train_geometries': len(grouped), 'heldout_used_for_training_or_selection': False,
               'model_checkpoint_sha256': p.file_hash(final / 'best.pt')}
    e.write_json(final / 'summary.json', summary)
    e.write_json(final / 'history.json', history)
    e.write_csv(final / 'history.csv', history)
    save_learning_plot(history, final / 'learning.png', fold['fold_id'], seed)
    e._commit(final, remote / 'final')
    del model, optimizer
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return final


def save_learning_plot(history, path, fold_id, seed):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 4))
    for metric in ['train_nll', 'validation_nll']:
        ax.plot([row['epoch'] for row in history], [row[metric] for row in history], label=metric)
    ax.set(title=f'{fold_id} · seed={seed} · NLL de imitación', xlabel='Época', ylabel='NLL por decisión')
    ax.legend(); fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def train_crossfit(instances, spec, config, seeds, corpus, work_root, drive_root, bank_id,
                   bank_sha256, evaluator_sha256, device='auto'):
    require_colab()
    validate_training_config(config)
    if (not seeds or any(type(seed) is not int or seed < 0 for seed in seeds)
            or len(set(seeds)) != len(seeds)):
        raise ValueError('Semillas de entrenamiento vacías, repetidas o inválidas.')
    if evaluator_sha256 != p.file_hash(p.__file__):
        raise ValueError('Evaluador de entrenamiento distinto del módulo sellado activo.')
    device = choose_device(device)
    folds = make_geometry_folds(instances)
    identity = {'config': config, 'seeds': seeds, 'folds': folds,
                'corpus_id': Path(corpus).name, 'bank_sha256': bank_sha256,
                'evaluator_sha256': evaluator_sha256, 'source_hashes': _source_hashes()}
    training_id = 'attention03_' + p.digest(identity)[:20]
    root = Path(work_root) / 'supervised_training' / bank_id / training_id
    remote = Path(drive_root) / 'supervised_training' / bank_id / training_id
    print('TRAINING_ID =', training_id, '| dispositivo:', device, flush=True)
    if e._recover(root / 'meta', remote / 'meta'):
        if e.read_json(root / 'meta/identity.json') != identity:
            raise ValueError('Identidad de entrenamiento inconsistente.')
    else:
        e.write_json(root / 'meta/identity.json', identity)
        e.write_json(root / 'meta/folds.json', folds)
        for name in _source_hashes():
            target = root / 'meta/code' / name
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(Path(__file__).with_name(name).read_bytes())
        e._commit(root / 'meta', remote / 'meta')
    models = []
    for fold in folds:
        for seed in seeds:
            relative = Path(fold['fold_id']) / f'seed_{seed}'
            attempt_started = time.perf_counter()
            try:
                final = train_fold(instances, spec, config, seed, fold, corpus,
                                   root / relative, remote / relative, device)
            except Exception as exc:
                failure = root / 'failures' / (fold['fold_id'] + f'_seed_{seed}_' + uuid.uuid4().hex[:8])
                e.write_json(failure / 'failure.json', {'fold': fold, 'seed': seed, 'training_id': training_id,
                             'error_type': type(exc).__name__, 'error': str(exc),
                             'observed_call_wall_seconds_including_io': time.perf_counter() - attempt_started,
                             'completed_checkpoints_preserved': True})
                e.write_json(failure / 'environment.json', p.environment_report())
                e._commit(failure, remote / 'failures' / failure.name)
                raise
            models.append({**e.read_json(final / 'summary.json'), 'relative_directory': str(relative / 'final'),
                           'bundle_sha256': p.verify_bundle(final)})
    # Toda selección terminó antes de que el notebook invoque evaluate sobre heldout.
    if not e._recover(root / 'index', remote / 'index'):
        _, _, provenance = load_fold_examples(corpus, instances, folds[0])
        e.write_json(root / 'index/models.json', models)
        e.write_json(root / 'index/offline_costs.json', {'offline_cost_id': training_id,
                     'teacher_corpus_id': provenance['corpus_id'],
                     'teacher_costs_unique': provenance['all_teacher_costs'],
                     'training_seconds_unique': sum(row['training_seconds'] for row in models),
                     'validation_seconds_included_in_training': sum(row['validation_seconds'] for row in models),
                     'models_trained': len(models), 'geometry_count': len(folds),
                     'summary_times_must_not_be_summed_across_policies_or_scenarios': True})
        e.write_csv(root / 'index/training_summary.csv', [{k: v for k, v in row.items()
                                                         if not isinstance(v, (dict, list))} for row in models])
        e._commit(root / 'index', remote / 'index')
    if e.read_json(root / 'index/models.json') != models:
        raise ValueError('Inventario de modelos distinto de los checkpoints.')
    return root


def make_solver(training_root, instances, spec, device='auto', expected_bank_sha256=None):
    """Carga sólo el modelo que dejó fuera la geometría de la instancia solicitada."""
    root, device = Path(training_root), choose_device(device)
    p.verify_bundle(root / 'meta')
    identity = e.read_json(root / 'meta/identity.json')
    if expected_bank_sha256 is not None and identity['bank_sha256'] != expected_bank_sha256:
        raise ValueError('Entrenamiento vinculado a otro banco.')
    p.verify_bundle(root / 'index')
    index = e.read_json(root / 'index/models.json')
    registry, provenance = {}, {}
    for model in index:
        final = root / model['relative_directory']
        if p.verify_bundle(final) != model['bundle_sha256'] or p.file_hash(final / 'best.pt') != model['model_checkpoint_sha256']:
            raise ValueError('Checkpoint de modelo modificado.')
        if model['traffic'] != asdict(spec):
            raise ValueError('Modelo entrenado con otro tráfico conocido.')
        fold = model['fold']
        validate_fold(fold, instances)
        for x in instances:
            if x.provenance['geometry_group_sha256'] not in fold['heldout_geometry_groups']:
                continue
            key = f'{x.sha}:{model["training_seed"]}'
            if key in registry:
                raise ValueError('Dos modelos compiten por una observación heldout.')
            registry[key] = (final, model)
            provenance[key] = {key_: model[key_] for key_ in ['training_seed', 'training_seconds',
                'validation_seconds', 'teacher_seconds', 'model_checkpoint_sha256', 'checkpoint_selection', 'training_scope']}
            provenance[key].update(training_id=root.name, fold_id=fold['fold_id'],
                train_geometry_groups=fold['train_geometry_groups'],
                validation_geometry_groups=fold['validation_geometry_groups'],
                heldout_geometry_groups=fold['heldout_geometry_groups'], offline_cost_id=root.name,
                offline_manifest_sha256=p.verify_bundle(root / 'index'))

    def solver(problem, seed, config, output_dir):
        started = time.perf_counter()
        key = f'{problem.x.sha}:{seed}'
        if key not in registry:
            raise ValueError('No existe modelo fuera de geometría para esta instancia/semilla.')
        final, record = registry[key]
        if p.file_hash(final / 'best.pt') != record['model_checkpoint_sha256']:
            raise ValueError('Checkpoint cambió antes de inferencia.')
        model = AttentionPolicy(record['model_config']).to(device)
        model.load_state_dict(_load_checkpoint(final / 'best.pt'))
        model.eval(); synchronize(device)
        loading = time.perf_counter() - started
        remaining = config['seconds'] - loading
        try:
            if remaining <= 0:
                result = problem.result(None, 'time_limit_model_load', loading, [])
            else:
                result = solve_attention(problem, seed, {**config, 'seconds': remaining}, model, device)
            result['metadata'].update(model_load_seconds=loading,
                                      model_checkpoint_sha256=record['model_checkpoint_sha256'],
                                      fold_id=record['fold']['fold_id'])
            synchronize(device)
            result['solve_seconds'] = time.perf_counter() - started
            return result
        finally:
            del model
            if device.type == 'cuda':
                torch.cuda.empty_cache()
    return solver, {'am_supervised': provenance}


Writing svrplab/supervised.py


In [11]:
%%writefile svrplab/pomo.py
"""04: REINFORCE multistart propio inspirado en POMO; ejecutar sólo en Colab.

Reutiliza AttentionPolicy y sus máscaras conservadoras sin etiquetas, escenarios
ni certificados. No reproduce el POMO original ni RL4CO: ventanas y vehículos
heterogéneos no garantizan simetría entre primeros clientes. Los rollouts del
grupo comparten una codificación y se decodifican secuencialmente.

POMO: Kwon et al., https://arxiv.org/html/2010.16011v3, secciones 4.1–4.3.
La primera acción impuesta queda fuera del logprob y el baseline es la media
detached del costo del grupo de la misma instancia. Se minimiza costo, de modo
que loss = mean((cost - mean(cost)).detach() * sum(logprob)).

Adaptaciones del proyecto: si un vehículo inicial no admite ningún cliente se
cierran vehículos vacíos sólo cuando la máscara común lo permite. Se fuerza un
cliente del primer vehículo con clientes admisibles; no se exploran prefijos
vacíos alternativos cuando éste sí admite clientes. Un único inicio se duplica
para dos continuaciones muestreadas en entrenamiento. Un bloqueo sin acciones
produce loss cero y se registra, sin afirmar infactibilidad de la instancia.
"""
from __future__ import annotations

from collections.abc import Mapping
from dataclasses import dataclass
import math
import time

import numpy as np
import torch

from .attention import (_State, _TOL, _expired, _prepare, _scales, _sync, _tensors,
                        MASK_VERSION, policy_completion_search)
from .search import SearchProblem


IMPLEMENTATION = 'project_reinforce_pomo_v2'
POMO_DEFAULTS = {'n_starts': 8}
INFERENCE_DEFAULTS = {'n_starts': 16, 'sampling_rounds': 1, 'permute_samples': False,
                      'all_starts': False, 'completion_search_nodes': 2048}
PENALTY = 'bounded_unserved_then_partial_travel_v1'


def validate_pomo_params(params=None):
    """Configuración de grupos de entrenamiento/validación; retorna copia normalizada."""
    if params is not None and not isinstance(params, Mapping):
        raise ValueError('Los parámetros POMO deben ser un mapping.')
    unknown = set(params or {}) - set(POMO_DEFAULTS)
    if unknown:
        raise ValueError(f'Parámetros POMO desconocidos: {sorted(unknown)}')
    result = POMO_DEFAULTS | dict(params or {})
    starts = result['n_starts']
    if isinstance(starts, (bool, np.bool_)) or not isinstance(starts, (int, np.integer)) or starts < 2:
        raise ValueError('n_starts debe ser entero >=2 para el baseline compartido.')
    return {'n_starts': int(starts)}


def _inference_config(config):
    if config is not None and not isinstance(config, Mapping):
        raise ValueError('La configuración de inferencia debe ser un mapping.')
    allowed = set(INFERENCE_DEFAULTS) | {'seconds'}
    unknown = set(config or {}) - allowed
    if unknown:
        raise ValueError(f'Parámetros de inferencia desconocidos: {sorted(unknown)}')
    result = {'seconds': 120.} | INFERENCE_DEFAULTS | dict(config or {})
    result.update(validate_pomo_params({'n_starts': result['n_starts']}))
    seconds, rounds = result['seconds'], result['sampling_rounds']
    if (isinstance(seconds, (bool, np.bool_)) or not isinstance(seconds, (int, float, np.number))
            or not math.isfinite(float(seconds)) or seconds < 0):
        raise ValueError('seconds debe ser finito y no negativo.')
    if isinstance(rounds, (bool, np.bool_)) or not isinstance(rounds, (int, np.integer)) or rounds < 0:
        raise ValueError('sampling_rounds debe ser entero no negativo.')
    if type(result['permute_samples']) is not bool:
        raise ValueError('permute_samples debe ser booleano.')
    if type(result['all_starts']) is not bool:
        raise ValueError('all_starts debe ser booleano.')
    if type(result['completion_search_nodes']) is not int or result['completion_search_nodes'] < 0:
        raise ValueError('completion_search_nodes debe ser entero no negativo.')
    return result | {'seconds': float(seconds), 'sampling_rounds': int(rounds)}


def _cost_scale(problem):
    """Todo recorrido completo tiene <= n+K arcos y costo <= U <= S."""
    upper = (problem.n + problem.vehicles) * float(np.max(problem.travel))
    if not math.isfinite(upper) or upper < 0 or bool((problem.travel < 0).any()):
        raise ValueError('Se requiere una matriz conservadora finita y no negativa.')
    return max(1., upper)


def _starting_customers(problem):
    """Prefijo vacío legal y clientes originales admisibles, sin consultar el modelo."""
    state = _State(problem)
    order, scales = np.arange(problem.n + 1), _scales(problem)
    while not state.done:
        blocked = state.inputs(order, scales)[2]
        customers = np.flatnonzero(~blocked[1:]) + 1
        if len(customers):
            return state.vehicle, customers
        if blocked[0]:
            break
        state.advance(0)
    return state.vehicle, np.array([], dtype=np.int64)


def _select_starts(customers, limit, rng=None, duplicate_single=False):
    if not len(customers):
        return [None, None] if duplicate_single else [None]
    if len(customers) == 1 and duplicate_single:
        return [int(customers[0]), int(customers[0])]
    count = min(limit, len(customers))
    selected = customers[:count] if rng is None else rng.choice(customers, count, replace=False)
    return [int(customer) for customer in selected]


@dataclass
class _Rollout:
    routes: list | None
    logprob: torch.Tensor
    decisions: int
    cost: float
    unserved: int
    termination: str


def _state_cost(state, scale):
    """Costo normalizado completo <=1; bloqueado en (1, 2+1/(n+1)].

    Para m clientes pendientes: 1 + m/n + P/[S(n+1)], donde P es el viaje
    de rutas parciales cerradas al depósito, S=max(1,(n+K)*max(travel)).
    Como P<=S, servir un cliente domina cualquier cambio en P. El término
    sólo entrena/selecciona checkpoint; nunca es una métrica del evaluador.
    """
    problem = state.problem
    missing = int((~state.visited).sum())
    if state.done and not missing:
        violation, travel = problem.score(state.routes)
        if violation > _TOL:
            raise ValueError('La construcción viola las máscaras conservadoras comunes.')
        return float(travel / scale), missing, True
    partial = sum(problem.route_stats(route, vehicle)['travel_minutes']
                  for vehicle, route in enumerate(state.routes))
    # La pausa por presupuesto también puede ocurrir después del último cliente;
    # ese rollout nunca entra al entrenamiento ni se devuelve como solución.
    cost = 1. + missing / max(1, problem.n) + partial / (scale * (problem.n + 1))
    return float(cost), missing, False


def _rollout(model, problem, device, prepared, prefix, first, mode, rng=None, deadline=None):
    """El inicio y cierres del prefijo son exógenos: ningún logprob para ellos."""
    encoded, order, inverse, scales = prepared
    state = _State(problem)
    for _ in range(prefix):
        state.advance(0)
    if first is not None:
        if bool(state.inputs(order, scales)[2][int(inverse[first])]):
            raise ValueError('Primer cliente POMO no admisible.')
        state.advance(first)
    terms, termination = [], 'complete'
    action_generator = None
    if mode == 'sample' and isinstance(rng, np.random.Generator):
        action_generator = torch.Generator(device=device).manual_seed(int(rng.integers(0, 2**63 - 1)))
    while not state.done:
        if _expired(deadline, device):
            termination = 'time_limit'
            break
        context, action, blocked = _tensors(state, order, scales, device)
        if bool(blocked.all()):
            termination = 'dead_end'
            break
        logits = model.next_logits(encoded, int(inverse[state.node]), context, action, blocked)
        if not bool(torch.isfinite(logits[~blocked]).all()):
            raise FloatingPointError('Logits POMO no finitos.')
        logprobs = torch.log_softmax(logits, dim=-1)
        if mode == 'greedy':
            local = int(torch.argmax(logits).item())
        elif mode == 'sample' and isinstance(rng, np.random.Generator):
            # Sample on-device: do not copy the full action distribution to CPU.
            # Its dedicated seed comes from the checkpointed NumPy RNG.
            probabilities = torch.softmax(logits, dim=-1)
            local = int(torch.multinomial(probabilities, 1, generator=action_generator).item())
        else:
            raise ValueError('Use greedy o sample con un Generator NumPy explícito.')
        terms.append(logprobs[local])
        state.advance(int(order[local]))
    cost, missing, complete = _state_cost(state, _cost_scale(problem))
    # Mantiene backward definido incluso en un grupo bloqueado antes de decidir.
    logprob = encoded.sum() * 0.
    if terms:
        logprob = logprob + torch.stack(terms).sum()
    if complete:
        termination = 'complete'
    return _Rollout(state.routes if complete else None, logprob, len(terms), cost, missing, termination)


def group_loss(model, problem, device, rng, params=None, node_order=None):
    """REINFORCE por instancia; caller controla train/eval, backward y optimizer.

    Selecciona primeros clientes sin reemplazo y muestrea sus continuaciones.
    Dropout debe ser cero: toda aleatoriedad de acciones procede de rng, cuyo
    estado debe conservarse en el checkpoint. No cambia datos ni flota.
    """
    effective = validate_pomo_params(params)
    if not isinstance(rng, np.random.Generator):
        raise ValueError('rng debe ser un Generator NumPy explícito.')
    if model.config.dropout != 0:
        raise ValueError('REINFORCE POMO requiere dropout=0 para la política de muestreo.')
    prefix, customers = _starting_customers(problem)
    starts = _select_starts(customers, effective['n_starts'], rng, duplicate_single=True)
    prepared = _prepare(model, problem, device, node_order)
    rollouts = [_rollout(model, problem, device, prepared, prefix, first, 'sample', rng)
                for first in starts]
    logprobs = torch.stack([rollout.logprob for rollout in rollouts])
    costs = torch.as_tensor([rollout.cost for rollout in rollouts], dtype=logprobs.dtype, device=device)
    baseline = costs.mean().detach()
    advantages = (costs - baseline).detach()
    loss = (advantages * logprobs).mean()
    if not bool(torch.isfinite(loss)):
        raise FloatingPointError('Loss REINFORCE no finita; detener entrenamiento.')
    complete_count = sum(rollout.routes is not None for rollout in rollouts)
    scale = _cost_scale(problem)
    return loss, {
        'decisions': sum(rollout.decisions for rollout in rollouts),
        'mean_cost': float(costs.mean().detach().cpu()),
        'mean_penalized_cost_minutes': float(costs.mean().detach().cpu()) * scale,
        'normalization_scale_minutes': scale,
        'complete_fraction': complete_count / len(rollouts),
        'complete_count': complete_count, 'rollout_count': len(rollouts),
        'forced_starts': starts, 'leading_empty_vehicles': prefix,
        'costs': [rollout.cost for rollout in rollouts],
        'advantages': advantages.cpu().tolist(),
        'zero_advantage_group': not bool((advantages != 0).any()),
        'unserved': [rollout.unserved for rollout in rollouts],
        'baseline': float(baseline.cpu()), 'penalty': PENALTY,
    }


def validation_score(model, problem, device, params=None):
    """Mejor costo de greedy multistart determinista; no usa semillas/escenarios.

    Los primeros IDs admisibles fijan el subconjunto cuando hay más que n_starts.
    Una sola alternativa se evalúa una vez; replicarla no cambiaría el greedy.
    """
    effective = validate_pomo_params(params)
    prefix, customers = _starting_customers(problem)
    starts = _select_starts(customers, effective['n_starts'])
    was_training = model.training
    model.eval()
    try:
        with torch.no_grad():
            prepared = _prepare(model, problem, device, None)
            rollouts = [_rollout(model, problem, device, prepared, prefix, first, 'greedy')
                        for first in starts]
        complete = sum(rollout.routes is not None for rollout in rollouts)
        return {'score': min(rollout.cost for rollout in rollouts),
                'complete_fraction': complete / len(rollouts), 'rollout_count': len(rollouts),
                'best_complete': complete > 0, 'forced_starts': starts,
                'normalization_scale_minutes': _cost_scale(problem), 'penalty': PENALTY}
    finally:
        model.train(was_training)


def solve_pomo(problem: SearchProblem, seed, config, model, device):
    """Greedy multistart + rondas de muestreo, con presupuesto único compartido.

    Sólo devuelve rutas completas. El límite incluye codificación, selección de
    inicios, construcción y validación; la carga de pesos pertenece al wrapper.
    """
    if isinstance(seed, (bool, np.bool_)) or not isinstance(seed, (int, np.integer)) or seed < 0:
        raise ValueError('La semilla debe ser un entero no negativo.')
    effective = _inference_config(config)
    device = torch.device(device)
    _sync(device)
    started = time.perf_counter()
    deadline = started + effective['seconds']
    rng = np.random.default_rng(int(seed))
    best, best_score, history = None, (math.inf, math.inf), []
    attempts = completed = failed = 0
    termination = 'round_limit'
    completion_search = {'termination': 'not_needed', 'expanded_nodes': 0, 'is_policy_sample': False}
    eligible_starts = starts_scheduled = 0
    was_training = model.training
    model.eval()
    try:
        with torch.inference_mode():
            for round_index in range(effective['sampling_rounds'] + 1):
                if _expired(deadline, device):
                    termination = 'time_limit'
                    break
                mode = 'greedy' if round_index == 0 else 'sample'
                prefix, customers = _starting_customers(problem)
                eligible_starts = len(customers)
                limit = len(customers) if effective['all_starts'] else effective['n_starts']
                starts = _select_starts(customers, limit, rng if mode == 'sample' else None)
                starts_scheduled += len(starts)
                order = (rng.permutation(np.arange(1, problem.n + 1))
                         if mode == 'sample' and effective['permute_samples'] else None)
                prepared = _prepare(model, problem, device, order)
                for first in starts:
                    if _expired(deadline, device):
                        termination = 'time_limit'
                        break
                    attempts += 1
                    rollout = _rollout(model, problem, device, prepared, prefix, first,
                                       mode, rng=rng, deadline=deadline)
                    if rollout.routes is None:
                        failed += 1
                    else:
                        completed += 1
                        score = problem.score(rollout.routes)
                        if score < best_score:
                            best, best_score = rollout.routes, score
                            _sync(device)
                            history.append({'iteration': attempts - 1,
                                            'elapsed_seconds': time.perf_counter() - started,
                                            'violation_minutes': float(score[0]), 'travel_minutes': float(score[1]),
                                            'phase': mode, 'round': round_index, 'first_customer': first})
                    if _expired(deadline, device):
                        termination = 'time_limit'
                        break
                if termination == 'time_limit':
                    break
            if best is None and effective['completion_search_nodes'] and not _expired(deadline, device):
                best, completion_search = policy_completion_search(
                    model, problem, device, deadline, effective['completion_search_nodes'])
                if best is not None:
                    best_score = problem.score(best)
                    history.append({'iteration': attempts, 'elapsed_seconds': time.perf_counter() - started,
                                    'violation_minutes': float(best_score[0]), 'travel_minutes': float(best_score[1]),
                                    'phase': 'policy_completion_search'})
    finally:
        model.train(was_training)
    if best is None and completion_search['termination'] == 'not_needed':
        completion_search['termination'] = ('no_remaining_budget' if effective['completion_search_nodes'] else 'disabled')
    if _expired(deadline, device):
        termination = 'time_limit'
    _sync(device)
    elapsed = time.perf_counter() - started
    result = problem.result(
        best, 'feasible' if best is not None else 'no_solution', elapsed, history,
        method='pomo', seed=int(seed), implementation=IMPLEMENTATION,
        model_config=model.config.to_dict(), effective_config=effective,
        termination=termination, attempts=attempts, completed_decodes=completed, failed_decodes=failed,
        raw_decode_complete_fraction=completed / attempts if attempts else None,
        completion_search=completion_search, mask_version=MASK_VERSION,
        eligible_first_customers=eligible_starts, starts_scheduled=starts_scheduled,
        all_starts_completed=(effective['all_starts'] and eligible_starts > 0
                              and attempts == starts_scheduled and termination != 'time_limit'),
        solution_origin='policy_completion_search' if completion_search['termination'] == 'feasible_found' else
                        'raw_policy_decode' if best is not None else 'none',
        inference_device=str(device), budget_includes_construction=True,
        budget_includes_encoding=True, budget_overrun_seconds=max(0., elapsed - effective['seconds']),
        time_limit_granularity='synchronized_decoder_step_or_completed_validation',
        search_objective='conservative_upper_travel_minutes',
        nominal_scope='conservative_upper_mean_matrix_not_stochastic_feasibility',
        construction=('policy_ordered_depth_first_completion' if completion_search['termination'] == 'feasible_found'
                      else 'forced_feasible_first_customer_without_certificates_or_repair'),
        no_solution_scope='decoder_dead_end_or_budget_not_infeasibility_proof',
        seed_reproducibility='same_environment_and_round_cutoff; wall_time_cutoff_may_differ',
        training_included_in_solve_seconds=False)
    _sync(device)
    elapsed = time.perf_counter() - started
    result['solve_seconds'] = elapsed
    result['metadata']['budget_overrun_seconds'] = max(0., elapsed - effective['seconds'])
    return result


Writing svrplab/pomo.py


In [12]:
%%writefile svrplab/checkpoint_diagnostics.py
"""Cruce 2×2 de checkpoints históricos 03/04; sólo inferencia en Google Colab.

No reentrena, selecciona épocas, cambia corpus ni incorpora sus filas a 06.
Separa señal de entrenamiento de procedimiento de inferencia; uniforme es un
control adicional. Los tiempos miden presupuestos máximos, no trabajo idéntico.
"""
from dataclasses import asdict
from pathlib import Path
import math
import time

import torch

from . import pilot as p, experiments as e
from .attention import AttentionPolicy, UniformMaskedPolicy, solve_attention
from .pomo import solve_pomo
from .search import SearchProblem, OBJECTIVE
from .supervised import choose_device, synchronize, _runtime_signature
from .supervised_data import validate_fold


def _inside(root, relative):
    root, relative = Path(root), Path(relative)
    if relative.is_absolute() or '..' in relative.parts or not relative.parts:
        raise ValueError('Ruta relativa inválida en entrenamiento elegido.')
    path = (root / relative).resolve()
    if not path.is_relative_to(root.resolve()):
        raise ValueError('Ruta fuera del entrenamiento elegido.')
    return path


def _models(root, instances, spec, bank_sha, seeds, time_scale, signal):
    root = Path(root)
    meta_sha, index_sha = p.verify_bundle(root / 'meta'), p.verify_bundle(root / 'index')
    identity = e.read_json(root / 'meta/identity.json')
    prefix = {'supervised': 'attention03_', 'reinforcement': 'pomo04_'}[signal]
    if (identity['bank_sha256'] != bank_sha or root.name != prefix + p.digest(identity)[:20]
            or identity['evaluator_sha256'] != p.file_hash(p.__file__)
            or identity['seeds'] != seeds):
        raise ValueError('Identidad, banco, evaluador o semillas del entrenamiento incompatibles.')
    training_config = identity['config']
    recorded_scale = (training_config['teacher']['time_scale'] if signal == 'supervised'
                      else training_config['time_scale'])
    if recorded_scale != time_scale or (signal == 'reinforcement' and training_config['objective'] != OBJECTIVE):
        raise ValueError('El cruce debe conservar la escala y el objetivo conservador de entrenamiento.')
    if e.read_json(root / 'meta/folds.json') != identity['folds']:
        raise ValueError('Folds distintos de la identidad de entrenamiento.')
    for name, digest in identity['source_hashes'].items():
        if p.file_hash(_inside(root / 'meta/code', name)) != digest:
            raise ValueError('Fuente histórica distinta del hash declarado en entrenamiento.')
    for name in ['attention.py', 'search.py', *(['pomo.py'] if signal == 'reinforcement' else [])]:
        saved = root / 'meta/code' / name
        if not saved.is_file() or p.file_hash(saved) != p.file_hash(Path(__file__).with_name(name)):
            raise ValueError(f'Cambió {name}; el cruce histórico exige la implementación de inferencia original.')
    result = {}
    for record in e.read_json(root / 'index/models.json'):
        fold = record['fold']
        validate_fold(fold, instances)
        if (record['traffic'] != asdict(spec) or fold not in identity['folds']
                or record['model_config'] != training_config['model']
                or record['heldout_used_for_training_or_selection'] is not False):
            raise ValueError('Tráfico, arquitectura o fold del checkpoint incompatible con su identidad.')
        final = _inside(root, record['relative_directory'])
        if (p.verify_bundle(final) != record['bundle_sha256']
                or p.file_hash(final / 'best.pt') != record['model_checkpoint_sha256']):
            raise ValueError('Checkpoint histórico alterado.')
        p.verify_bundle(final.parent / 'meta')
        fold_identity = e.read_json(final.parent / 'meta/identity.json')
        if (p.digest(fold_identity) != record['identity_sha256'] or fold_identity['fold'] != fold
                or fold_identity['seed'] != record['training_seed'] or fold_identity['config'] != training_config):
            raise ValueError('Los pesos no corresponden a la identidad sellada del fold.')
        if e.read_json(final / 'summary.json') != {k: v for k, v in record.items()
                                                  if k not in ('relative_directory', 'bundle_sha256')}:
            raise ValueError('Resumen e índice de checkpoint discrepan.')
        for instance in instances:
            if instance.provenance['geometry_group_sha256'] in fold['heldout_geometry_groups']:
                key = (instance.sha, record['training_seed'])
                if key in result:
                    raise ValueError('Dos modelos para la misma geometría y semilla.')
                result[key] = (final, record)
    expected = {(x.sha, seed) for x in instances for seed in seeds}
    if set(result) != expected:
        raise ValueError('Inventario histórico distinto del plan prefijado; no omitir modelos fallidos.')
    return result, {'training_id': root.name, 'meta_sha256': meta_sha, 'index_sha256': index_sha}


def run_checkpoint_crossover(bank, roots, config, work_root, drive_root, device='cuda'):
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise RuntimeError('Inferencia diagnóstica exclusivamente en Google Colab.') from exc
    if set(roots) != {'supervised', 'reinforcement'} or config['role'] != 'development_diagnostic_only':
        raise ValueError('Se requieren ambos entrenamientos históricos y rol diagnóstico.')
    if config['analysis_mode'] != 'descriptive_only':
        raise ValueError('El cruce no habilita pruebas confirmatorias.')
    if (not config['seeds'] or len(set(config['seeds'])) != len(config['seeds'])
            or any(type(seed) is not int or seed < 0 for seed in config['seeds'])
            or type(config['time_scale']) is not int or config['time_scale'] <= 0
            or isinstance(config['seconds_per_attempt'], bool)
            or not math.isfinite(config['seconds_per_attempt']) or config['seconds_per_attempt'] <= 0
            or set(config['decoders']) != {'am', 'pomo'}
            or any(Path(roots[signal]).name != config['training_ids'][signal] for signal in roots)):
        raise ValueError('Semillas, presupuesto, decoders o entrenamientos distintos del plan diagnóstico.')
    bank, device = Path(bank), choose_device(device)
    if device.type != 'cuda':
        raise RuntimeError('El cruce requiere GPU en Colab para mantener el recurso declarado.')
    bank_sha = p.verify_bundle(bank)
    instances = p.load_bank(bank)
    bank_config = e.read_json(bank / 'config.json')
    spec = p.Traffic(**bank_config['traffic'])
    models, model_identities = {}, {}
    for signal in ('supervised', 'reinforcement'):
        models[signal], model_identities[signal] = _models(roots[signal], instances, spec, bank_sha,
            config['seeds'], config['time_scale'], signal)
    for key in models['supervised']:
        a, b = models['supervised'][key][1], models['reinforcement'][key][1]
        # Historical 03 used dropout=.1 and 04 dropout=0. This changes training,
        # not eval-mode topology: insisting on complete config equality blocks
        # the actual requested historical crossover. All other fields must match.
        inference_a = {k: v for k, v in a['model_config'].items() if k != 'dropout'}
        inference_b = {k: v for k, v in b['model_config'].items() if k != 'dropout'}
        if a['fold'] != b['fold'] or inference_a != inference_b:
            raise ValueError('El cruce exige arquitectura de inferencia y folds iguales; sólo admite dropout distinto en entrenamiento.')
    identity = {'schema_version': 'checkpoint-crossover-v1', 'bank_sha256': bank_sha,
                'models': model_identities, 'config': config,
                'sources': {name: p.file_hash(Path(__file__).with_name(name)) for name in
                            ('checkpoint_diagnostics.py', 'attention.py', 'pomo.py', 'search.py',
                             'supervised.py', 'supervised_data.py', 'experiments.py')},
                'evaluator_sha256': p.file_hash(p.__file__), 'runtime_signature': _runtime_signature(device),
                'dropout_scope': 'historical_training_difference_disabled_by_model_eval',
                'no_retraining_or_checkpoint_selection': True,
                'independent_unit': 'geometry_group_sha256', 'native_decoders_common_cap_not_equal_work': True}
    identifier = 'crossover03_' + p.digest(identity)[:20]
    root, remote = [Path(base) / 'checkpoint_diagnostics' / bank.name / identifier
                    for base in (work_root, drive_root)]
    print('DIAGNOSTIC_ID =', identifier, flush=True)
    if not e._recover(root / 'meta', remote / 'meta'):
        e.write_json(root / 'meta/identity.json', identity)
        e.write_json(root / 'meta/environment.json', p.environment_report())
        e._commit(root / 'meta', remote / 'meta')
    if e.read_json(root / 'meta/identity.json') != identity:
        raise ValueError('Identidad de reanudación incompatible.')
    rows, bundles = [], {}
    for x in sorted(instances, key=lambda item: item.n):
        for seed in config['seeds']:
            for signal in ('supervised', 'reinforcement', 'uniform'):
                final, record = models['supervised' if signal == 'uniform' else signal][(x.sha, seed)]
                for decoder, solve in (('am', solve_attention), ('pomo', solve_pomo)):
                    relative = f'{signal}/{decoder}/{x.sha}/seed_{seed}'
                    directory = root / relative
                    if not e._recover(directory, remote / relative):
                        started = time.perf_counter()
                        model = None
                        try:
                            problem = SearchProblem(x, spec, config['time_scale'])
                            if signal == 'uniform':
                                model = UniformMaskedPolicy(record['model_config']).to(device)
                            else:
                                # The preceding bundle + file checks are repeated before deserialization.
                                if p.file_hash(final / 'best.pt') != record['model_checkpoint_sha256']:
                                    raise ValueError('Checkpoint cambió antes de cargarlo.')
                                model = AttentionPolicy(record['model_config']).to(device)
                                model.load_state_dict(torch.load(final / 'best.pt', map_location='cpu', weights_only=True))
                            model.eval()
                            synchronize(device)
                            remaining = max(0., config['seconds_per_attempt'] - (time.perf_counter() - started))
                            settings = dict(config['decoders'][decoder], seconds=remaining)
                            solution = solve(problem, seed, settings, model, device) if remaining else problem.result(None, 'time_limit_model_load', time.perf_counter() - started, [])
                            synchronize(device)
                            score = None if solution['routes'] is None else problem.score(solution['routes'])
                            if solution['routes'] is not None:
                                p.validate_routes(x, solution['routes'])
                            row = {'signal': signal, 'decoder': decoder, 'solver_seed': seed,
                                   'instance_sha256': x.sha, 'geometry_group_sha256': x.provenance['geometry_group_sha256'],
                                   'n_customers': x.n, 'fold_id': record['fold']['fold_id'],
                                   'checkpoint_sha256': None if signal == 'uniform' else record['model_checkpoint_sha256'],
                                   'training_dropout': None if signal == 'uniform' else record['model_config']['dropout'],
                                   'dropout_active_at_inference': model.training,
                                   'objective_scope': OBJECTIVE, 'time_scale': config['time_scale'],
                                   'status': solution['status'], 'complete_feasible': score is not None and score[0] <= 1e-9,
                                   'conservative_travel_minutes': None if score is None else score[1],
                                   'search_violation_minutes': None if score is None else score[0],
                                   'seconds': time.perf_counter() - started,
                                   'cap_seconds': config['seconds_per_attempt'],
                                   'solution_origin': solution.get('metadata', {}).get('solution_origin'),
                                   'role': config['role']}
                        finally:
                            del model
                            torch.cuda.empty_cache()
                        # Unexpected technical failures stop; they never become scientific zeros.
                        e.write_json(directory / 'row.json', row)
                        e.write_json(directory / 'solution.json', solution)
                        e._commit(directory, remote / relative)
                    recorded = e.read_json(directory / 'row.json')
                    expected_row = {'signal': signal, 'decoder': decoder, 'solver_seed': seed,
                        'instance_sha256': x.sha, 'geometry_group_sha256': x.provenance['geometry_group_sha256'],
                        'n_customers': x.n, 'fold_id': record['fold']['fold_id'],
                        'objective_scope': OBJECTIVE, 'time_scale': config['time_scale'],
                        'cap_seconds': config['seconds_per_attempt'], 'role': config['role']}
                    if any(recorded.get(key) != value for key, value in expected_row.items()):
                        raise ValueError('Resultado diagnóstico recuperado de otra identidad o decoder.')
                    rows.append(recorded)
                    bundles[relative] = p.verify_bundle(directory)
                    print(x.n, seed, signal, decoder, rows[-1]['status'], flush=True)
    expected = len(instances) * len(config['seeds']) * 6
    if len(rows) != expected:
        raise ValueError('Cruce incompleto; no publicar tabla parcial como completo.')
    # Detect source changes during the crossover; no implicit migration to new weights.
    for signal in ('supervised', 'reinforcement'):
        _, final_identity = _models(roots[signal], instances, spec, bank_sha,
            config['seeds'], config['time_scale'], signal)
        if final_identity != model_identities[signal]:
            raise ValueError('El entrenamiento fuente cambió durante el cruce.')
    if p.verify_bundle(bank) != bank_sha:
        raise ValueError('Banco alterado durante el cruce.')
    if not e._recover(root / 'report', remote / 'report'):
        e.write_csv(root / 'report/crossover.csv', rows)
        e.write_json(root / 'report/bundles.json', bundles)
        e.write_json(root / 'report/protocol.json', identity)
        e.write_json(root / 'report/status.json', {'status': 'complete_diagnostic', 'attempts': expected,
                     'confirmatory': False, 'retrained': False, 'new_independent_geometries': 0})
        e._commit(root / 'report', remote / 'report')
    if (e.read_json(root / 'report/bundles.json') != bundles
            or e.read_json(root / 'report/protocol.json') != identity):
        raise ValueError('El reporte recuperado no corresponde a los intentos completos actuales.')
    return root


Writing svrplab/checkpoint_diagnostics.py


In [13]:
DIAGNOSTICS_CONFIG = {'schema_version': 'checkpoint-crossover-v1', 'role': 'development_diagnostic_only', 'analysis_mode': 'descriptive_only', 'seeds': [11, 29, 47], 'time_scale': 1000, 'seconds_per_attempt': 120.0, 'training_ids': {'supervised': 'attention03_166b68cd3de275cafbed', 'reinforcement': 'pomo04_cc0f02c7a2de361e89e4'}, 'decoders': {'am': {'n_samples': 16, 'permute_samples': False, 'completion_search_nodes': 2048}, 'pomo': {'n_starts': 16, 'sampling_rounds': 1, 'permute_samples': False, 'all_starts': False, 'completion_search_nodes': 2048}}}


In [14]:
import importlib
if str(RUNTIME) not in sys.path:
    sys.path.insert(0, str(RUNTIME))
for name in list(sys.modules):
    if name == 'svrplab' or name.startswith('svrplab.'):
        del sys.modules[name]
importlib.invalidate_caches()
from svrplab.checkpoint_diagnostics import run_checkpoint_crossover
ROOTS = {
    'supervised': DRIVE_ROOT / 'supervised_training' / BANK_ID / DIAGNOSTICS_CONFIG['training_ids']['supervised'],
    'reinforcement': DRIVE_ROOT / 'rl_training' / BANK_ID / DIAGNOSTICS_CONFIG['training_ids']['reinforcement']
}
DIAGNOSTIC = run_checkpoint_crossover(BANK, ROOTS, DIAGNOSTICS_CONFIG, WORK, DRIVE_ROOT, device='cuda')
print('Diagnóstico sellado:', DIAGNOSTIC)
import pandas as pd
from IPython.display import display
display(pd.read_csv(DIAGNOSTIC / 'report/crossover.csv', dtype={'solver_seed': str}))


DIAGNOSTIC_ID = crossover03_b909112516a592c33bcc
50 11 supervised am no_solution
50 11 supervised pomo no_solution
50 11 reinforcement am no_solution
50 11 reinforcement pomo no_solution
50 11 uniform am no_solution
50 11 uniform pomo no_solution
50 29 supervised am no_solution
50 29 supervised pomo no_solution
50 29 reinforcement am no_solution
50 29 reinforcement pomo no_solution
50 29 uniform am no_solution
50 29 uniform pomo no_solution
50 47 supervised am no_solution
50 47 supervised pomo no_solution
50 47 reinforcement am no_solution
50 47 reinforcement pomo no_solution
50 47 uniform am no_solution
50 47 uniform pomo no_solution
100 11 supervised am no_solution
100 11 supervised pomo no_solution
100 11 reinforcement am no_solution
100 11 reinforcement pomo feasible
100 11 uniform am no_solution
100 11 uniform pomo no_solution
100 29 supervised am no_solution
100 29 supervised pomo no_solution
100 29 reinforcement am feasible
100 29 reinforcement pomo feasible
100 29 uniform am no

,signal,decoder,solver_seed,instance_sha256,geometry_group_sha256,n_customers,fold_id,checkpoint_sha256,training_dropout,dropout_active_at_inference,objective_scope,time_scale,status,complete_feasible,conservative_travel_minutes,search_violation_minutes,seconds,cap_seconds,solution_origin,role
0,supervised,am,11,27f2c447bd200284b6b6b8dd98f917e3a283fec26aec64...,c97ac39c9cc9397d54db6243bb00bbd92f7c26e248b072...,50,geometry_03,d1d1c911451e8107daec6ea5748f83e2ba3a064e75dc36...,0.1,False,conservative_upper_travel_minutes_v1,1000,no_solution,False,NaN,NaN,8.951777,120.0,none,development_diagnostic_only
1,supervised,pomo,11,27f2c447bd200284b6b6b8dd98f917e3a283fec26aec64...,c97ac39c9cc9397d54db6243bb00bbd92f7c26e248b072...,50,geometry_03,d1d1c911451e8107daec6ea5748f83e2ba3a064e75dc36...,0.1,False,conservative_upper_travel_minutes_v1,1000,no_solution,False,NaN,NaN,7.532664,120.0,none,development_diagnostic_only
2,reinforcement,am,11,27f2c447bd200284b6b6b8dd98f917e3a283fec26aec64...,c97ac39c9cc9397d54db6243bb00bbd92f7c26e248b072...,50,geometry_03,a4cd4ce8c1426bc2613598a2ffcec316cac7633e113ef4...,0.0,False,conservative_upper_travel_minutes_v1,1000,no_solution,False,NaN,NaN,6.118910,120.0,none,development_diagnostic_only
3,reinforcement,pomo,11,27f2c447bd200284b6b6b8dd98f917e3a283fec26aec64...,c97ac39c9cc9397d54db6243bb00bbd92f7c26e248b072...,50,geometry_03,a4cd4ce8c1426bc2613598a2ffcec316cac7633e113ef4...,0.0,False,conservative_upper_travel_minutes_v1,1000,no_solution,False,NaN,NaN,9.622340,120.0,none,development_diagnostic_only
4,uniform,am,11,27f2c447bd200284b6b6b8dd98f917e3a283fec26aec64...,c97ac39c9cc9397d54db6243bb00bbd92f7c26e248b072...,50,geometry_03,NaN,NaN,False,conservative_upper_travel_minutes_v1,1000,no_solution,False,NaN,NaN,2.456364,120.0,none,development_diagnostic_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,supervised,pomo,47,8a8356445234902ced061220b435c5d90c942c32488a61...,82848d94ab655d9e6e51c83452358ad34d7cb90d4b17cc...,300,geometry_02,ceef8355c381b1ddd6d058869a1e13d40984dd5795ab43...,0.1,False,conservative_upper_travel_minutes_v1,1000,no_solution,False,NaN,NaN,38.734941,120.0,none,development_diagnostic_only
68,reinforcement,am,47,8a8356445234902ced061220b435c5d90c942c32488a61...,82848d94ab655d9e6e51c83452358ad34d7cb90d4b17cc...,300,geometry_02,3ad4caeb66a6d4e777d764bffb9466ba34786c3a84b11f...,0.0,False,conservative_upper_travel_minutes_v1,1000,feasible,True,56603.007,0.0,19.257045,120.0,raw_policy_decode,development_diagnostic_only
69,reinforcement,pomo,47,8a8356445234902ced061220b435c5d90c942c32488a61...,82848d94ab655d9e6e51c83452358ad34d7cb90d4b17cc...,300,geometry_02,3ad4caeb66a6d4e777d764bffb9466ba34786c3a84b11f...,0.0,False,conservative_upper_travel_minutes_v1,1000,feasible,True,56600.492,0.0,36.917837,120.0,raw_policy_decode,development_diagnostic_only
70,uniform,am,47,8a8356445234902ced061220b435c5d90c942c32488a61...,82848d94ab655d9e6e51c83452358ad34d7cb90d4b17cc...,300,geometry_02,NaN,NaN,False,conservative_upper_travel_minutes_v1,1000,no_solution,False,NaN,NaN,12.813312,120.0,none,development_diagnostic_only


Interpretación: cambiar únicamente el decoder dentro de una misma señal identifica
dependencia del procedimiento de inferencia. Cambiar la señal entre checkpoints también
cambia corpus, pérdida, selección, dropout (0.1 supervisado y 0.0 refuerzo) y cómputo de entrenamiento.
El dropout se desactiva con `model.eval()` en ambos; los demás campos de arquitectura
y la escala/objetivo conservador deben coincidir. Esto no prueba por sí solo una
superioridad causal del refuerzo. Uniforme calibra la contribución de máscara/buscador.
Canonización de etiquetas es una intervención posterior de desarrollo; no se aplicará
retroactivamente ni se escogerá una versión con resultados de prueba final.
